<a href="https://colab.research.google.com/github/sigma-asif/explainable-transport-mode-choice/blob/main/2_qwen3_llm_explanations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Transport Mode — Standalone Qwen3-8B Explanation Notebook

This notebook is **downstream only**. It does not retrain Random Forest, rerun RandomOverSampler, change model probabilities, recalculate SHAP, or choose a new dominant role.

It reads the fixed artifacts produced by `Q1_Transport_Mode_FINAL_ML_RF_Standalone.ipynb`:

1. 33-conceptual-predictor respondent data;
2. frozen RF OOF class predictions/probabilities;
3. predicted-class OOF conceptual SHAP values;
4. pre-computed Local Top-2 dominant-role output.

The observed target and prediction correctness are excluded from the LLM payload. Qwen3-8B is used only to verbalize supplied model evidence. A strict automated audit checks factual/structural fidelity and separately records semantic overclaim flags.




In [ ]:
# ============================================================
# RUNTIME DEPENDENCIES — DO NOT UPGRADE NUMPY/SCIPY HERE
# ============================================================
import sys, subprocess
from importlib import metadata

PACKAGES = ["transformers", "accelerate", "bitsandbytes"]
missing = []
for pkg in PACKAGES:
    try:
        metadata.version(pkg)
    except metadata.PackageNotFoundError:
        missing.append(pkg)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Python:", sys.version.split()[0])
for pkg in PACKAGES:
    try:
        print(f"{pkg:15s}: {metadata.version(pkg)}")
    except Exception:
        pass


Python: 3.13.15
transformers   : 5.16.1
accelerate     : 1.14.0
bitsandbytes   : 0.50.2


In [ ]:
# ============================================================
# GOOGLE DRIVE + GPU CHECK
# ============================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Mounted at /content/drive
CUDA available: True
GPU: Tesla T4


## Build the fixed evidence payload

This cell joins the saved OOF predictions, conceptual predicted-class SHAP, respondent values and the pre-computed Local Top-2 dominant role. It also performs respondent alignment and internal consistency checks before creating the LLM-ready JSON.


In [ ]:
# ============================================================
# FINAL AGENTIC-AI EVIDENCE BUILDER
# FINAL 43-ENCODED RF + 33-CONCEPTUAL SHAP PIPELINE
#
# IMPORTANT:
# ------------------------------------------------------------
# THIS SCRIPT DOES NOT:
#   - retrain Random Forest
#   - rerun RandomOverSampler
#   - recalculate SHAP
#
# IT LOADS THE FINAL FIXED OOF OUTPUTS FROM:
#
#   1. Cleaned_Data_Final_Preprocessed.csv
#   2. oof_respondent_predictions.csv
#   3. oof_local_shap_predicted_class.csv
#   4. dominant_role_respondent_level_ros_rf.csv
#
# Therefore every Agentic-AI explanation is grounded in
# exactly the SAME OOF prediction + local SHAP used in the
# final ROS + RF analysis.
#
#
# DOMINANT ROLE:
# ------------------------------------------------------------
# The dominant-role label is NOT selected by the LLM.
#
# The fixed pre-computed label:
#
#     Dominant_Role_LocalTop2
#
# is loaded from the separate dominant-role pipeline.
#
# For explainability only, this cell retrieves the two
# respondent-level features within that role having the
# largest absolute predicted-class local SHAP values.
#
# Their signed SHAP values are retained.
#
# The Top-2 absolute SHAP sum is:
#
#     |SHAP_1| + |SHAP_2|
#
# An internal consistency check verifies that the supplied
# pre-computed dominant role has the maximum Local Top-2
# absolute SHAP score among:
#
#     Economic
#     Safety
#     Psychological
#     Other
#
# This verification does NOT change the supplied role.
#
#
# PIPELINE
# ------------------------------------------------------------
# Final 33-conceptual-predictor dataset
#           +
# Final OOF RF predictions/probabilities
#           +
# Final predicted-class local OOF SHAP
#           ↓
# Respondent alignment verification
#           ↓
# Feature decoding
#           ↓
# Top supporting SHAP evidence
# Top opposing SHAP evidence
#           ↓
# Runner-up probability + probability margin
#           ↓
# Pre-computed Local Top-2 dominant role
#           +
# Exact Top-2 contributors within supplied dominant role
#           +
# Exact Local Top-2 absolute SHAP sum
#           ↓
# LLM-ready JSON / JSONL / audit CSV
#
# TRUE CLASS:
# ------------------------------------------------------------
# Kept in audit CSV only.
# NOT supplied to the LLM-ready JSON to reduce hindsight bias.
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import json
import re

import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

DATA_FILE = (
    "/content/drive/MyDrive/Research/Transport Mode/"
    "FINAL TRANSPORT MODE/Q1 PIPELINE V2/preprocessing/"
    "Cleaned_Data_Final_Preprocessed_Q1_Conceptual33.csv"
)


FINAL_RF_METRIC_DIR = (
    "/content/drive/MyDrive/Research/Transport Mode/"
    "FINAL TRANSPORT MODE/Q1 PIPELINE V2/final_rf/metrics"
)


PREDICTION_FILE = os.path.join(
    FINAL_RF_METRIC_DIR,
    "oof_respondent_predictions.csv"
)


LOCAL_SHAP_FILE = os.path.join(
    FINAL_RF_METRIC_DIR,
    "oof_local_shap_predicted_class.csv"
)


DOMINANT_ROLE_FILE = (
    "/content/drive/MyDrive/Research/Transport Mode/"
    "FINAL TRANSPORT MODE/Q1 PIPELINE V2/dominant_role/metrics/"
    "dominant_role_respondent_level_ros_rf.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/Research/Transport Mode/"
    "FINAL TRANSPORT MODE/Q1 PIPELINE V2/Agentic AI"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 2. SETTINGS
# ============================================================

TARGET = "Transport_Grouped"


# Strongest positive SHAP features supplied to LLM.
TOP_POSITIVE = 5


# Strongest negative SHAP features supplied to LLM.
TOP_NEGATIVE = 5


# Ignore extremely small SHAP values ONLY for the general
# supporting/opposing evidence lists.
#
# IMPORTANT:
# This threshold is NOT applied to the dominant-role Top-2
# contributors, because the Local Top-2 dominant-role method
# itself does not use this threshold.
MIN_ABS_SHAP = 0.005


# Number of contributors defining the dominant-role score.
DOMINANT_ROLE_TOP_K = 2


ROLE_CATEGORIES = [
    "Economic",
    "Safety",
    "Psychological",
    "Other"
]


# ============================================================
# 3. LOAD FINAL DATA
# ============================================================

df = pd.read_csv(
    DATA_FILE
)


pred_df = pd.read_csv(
    PREDICTION_FILE
)


shap_df = pd.read_csv(
    LOCAL_SHAP_FILE
)


dominant_role_df = pd.read_csv(
    DOMINANT_ROLE_FILE
)


print("=" * 90)

print(
    "FINAL AGENTIC-AI EVIDENCE BUILDER "
    "— ROS + RF + OOF SHAP + LOCAL TOP-2 DOMINANT ROLE"
)

print("=" * 90)


print(
    "Dataset rows       :",
    len(df)
)


print(
    "Prediction rows    :",
    len(pred_df)
)


print(
    "Local SHAP rows    :",
    len(shap_df)
)


print(
    "Dominant-role rows :",
    len(dominant_role_df)
)


# ============================================================
# 4. FINAL FEATURE SET
# ============================================================

assert TARGET in df.columns


X = df.drop(
    columns=[TARGET]
).copy()


y_true = df[
    TARGET
].copy()


feature_names = list(
    X.columns
)


assert len(feature_names) == 33, (
    f"Expected 33 final predictors, "
    f"found {len(feature_names)}"
)


print(
    "\nFinal predictors:",
    len(feature_names)
)


for i, feature in enumerate(
    feature_names,
    start=1
):

    print(
        f"{i:02d}. {feature}"
    )


# ============================================================
# 5. FINAL EXPECTED FEATURES
# ============================================================

EXPECTED_FEATURES = [

    "Gender",

    "Age",

    "Monthly Income",

    "Average Expendature for transportation",

    "Importance of Cost in Mode Choice",

    "Effect of Cost on Mode Choice",

    "Daily Activity Influence  Mode choice",

    "Distance Influence on Mode Choice",

    "Importance of Safety in Mode Choice",

    "Mobility Barrier on Mode Choice",

    "Accident Memories On Mode Choice",

    "Environmental Impact on Mode Choice",

    "Trust level based on safety in Bus",

    "Trust level based on safety in CNG",

    "Trust level based on safety in Easybike",

    "Trust level based on safety in Rickshaw",

    "Trust level based on safety in Bike",

    "Trust level based on safety in Car",

    "Trust level based on safety in Cycle",

    "Mode_When_Depressed",

    "Mode_When_Happy",

    "Mode_When_Stressed",

    "Mode_When_Urgent",

    "Mode_In_Weather",

    "Occ_Agriculture",

    "Occ_Business_SelfEmployed",

    "Occ_Homemaker",

    "Occ_Professional",

    "Occ_Service_Labour",

    "Occ_Student",

    "TravelObj_Medical_Recreation",

    "TravelObj_Shopping",

    "TravelObj_Work"
]


assert set(
    feature_names
) == set(
    EXPECTED_FEATURES
), (
    "Feature mismatch between expected "
    "final model and dataset."
)


# ============================================================
# 6. CHECK OOF FILE STRUCTURE
# ============================================================

REQUIRED_PREDICTION_COLUMNS = {

    "row_index",

    "true_class",

    "predicted_class",

    "predicted_confidence",

    "correct"
}


missing_prediction_cols = (

    REQUIRED_PREDICTION_COLUMNS
    -
    set(pred_df.columns)
)


assert not missing_prediction_cols, (
    "Missing prediction columns: "
    f"{missing_prediction_cols}"
)


REQUIRED_SHAP_META_COLUMNS = {

    "row_index",

    "true_class",

    "predicted_class"
}


missing_shap_meta = (

    REQUIRED_SHAP_META_COLUMNS
    -
    set(shap_df.columns)
)


assert not missing_shap_meta, (
    "Missing SHAP metadata columns: "
    f"{missing_shap_meta}"
)


REQUIRED_DOMINANT_ROLE_COLUMNS = {

    "Respondent_ID",

    "Dominant_Role_LocalTop2"
}


missing_dominant_role_columns = (

    REQUIRED_DOMINANT_ROLE_COLUMNS
    -
    set(dominant_role_df.columns)
)


assert not missing_dominant_role_columns, (
    "Missing dominant-role columns: "
    f"{sorted(missing_dominant_role_columns)}"
)


# ============================================================
# 7. EXPECTED SHAP COLUMNS
# ============================================================

SHAP_COLUMNS = {

    feature:
        f"SHAP__{feature}"

    for feature
    in feature_names
}


missing_shap_features = [

    shap_col

    for shap_col
    in SHAP_COLUMNS.values()

    if shap_col
    not in shap_df.columns
]


assert not missing_shap_features, (
    "Missing local SHAP feature columns: "
    f"{missing_shap_features}"
)


# ============================================================
# 8. RESPONDENT-ID VALIDATION
# ============================================================

expected_indices = set(
    range(
        len(df)
    )
)


assert set(
    pred_df[
        "row_index"
    ].astype(int)
) == expected_indices, (
    "Prediction respondent indices "
    "do not match dataset."
)


assert set(
    shap_df[
        "row_index"
    ].astype(int)
) == expected_indices, (
    "SHAP respondent indices "
    "do not match dataset."
)


dominant_role_df[
    "Respondent_ID"
] = dominant_role_df[
    "Respondent_ID"
].astype(int)


assert set(
    dominant_role_df[
        "Respondent_ID"
    ]
) == expected_indices, (
    "Dominant-role respondent indices "
    "do not match dataset."
)


assert not dominant_role_df[
    "Respondent_ID"
].duplicated().any(), (
    "Duplicate Respondent_ID "
    "in dominant-role file."
)


assert not pred_df[
    "row_index"
].duplicated().any(), (
    "Duplicate respondent IDs "
    "in prediction file."
)


assert not shap_df[
    "row_index"
].duplicated().any(), (
    "Duplicate respondent IDs "
    "in SHAP file."
)


# ============================================================
# 9. SORT ALL INPUTS BY ORIGINAL ROW INDEX
# ============================================================

pred_df = (

    pred_df

    .sort_values(
        "row_index"
    )

    .reset_index(
        drop=True
    )
)


shap_df = (

    shap_df

    .sort_values(
        "row_index"
    )

    .reset_index(
        drop=True
    )
)


dominant_role_df = (

    dominant_role_df

    .sort_values(
        "Respondent_ID"
    )

    .set_index(
        "Respondent_ID"
    )
)


X = X.reset_index(
    drop=True
)


y_true = y_true.reset_index(
    drop=True
)


# ============================================================
# 10. CROSS-FILE ALIGNMENT CHECK
# ============================================================

assert np.all(

    y_true.astype(str).values

    ==

    pred_df[
        "true_class"
    ].astype(str).values

), (
    "True-class mismatch between dataset "
    "and OOF prediction file."
)


assert np.all(

    pred_df[
        "predicted_class"
    ].astype(str).values

    ==

    shap_df[
        "predicted_class"
    ].astype(str).values

), (
    "Predicted-class mismatch between "
    "OOF prediction and local SHAP files."
)


assert np.all(

    pred_df[
        "true_class"
    ].astype(str).values

    ==

    shap_df[
        "true_class"
    ].astype(str).values
)


print(
    "\nCross-file alignment: PASSED"
)


# ============================================================
# 11. CLASS NAMES
# ============================================================

classes = sorted(
    y_true.unique()
)


print(
    "\nClasses:",
    classes
)


# ============================================================
# 12. PROBABILITY COLUMN MAPPING
# ============================================================

def safe_class_name(
    class_name
):

    return (

        str(
            class_name
        )

        .replace(
            "/",
            "_"
        )

        .replace(
            " ",
            "_"
        )
    )


PROBABILITY_COLUMNS = {

    class_name:
        f"prob_{safe_class_name(class_name)}"

    for class_name
    in classes
}


print(
    "\nProbability columns:"
)


for class_name, column in (
    PROBABILITY_COLUMNS.items()
):

    print(
        f"{class_name:20s} -> {column}"
    )


for class_name, column in (
    PROBABILITY_COLUMNS.items()
):

    assert column in pred_df.columns, (
        f"Missing probability column: "
        f"{column}"
    )


prob_matrix = pred_df[

    list(
        PROBABILITY_COLUMNS.values()
    )

].to_numpy(
    dtype=float
)


assert np.allclose(

    prob_matrix.sum(
        axis=1
    ),

    1.0,

    atol=1e-6

), (
    "Class probabilities do not sum to 1."
)


print(
    "Probability validation: PASSED"
)


# ============================================================
# 13. ONE-HOT GROUP VALIDATION
# ============================================================

OCCUPATION_DUMMIES = {

    "Occ_Agriculture":
        "Agriculture",

    "Occ_Business_SelfEmployed":
        "Business / Self-employed",

    "Occ_Homemaker":
        "Homemaker",

    "Occ_Professional":
        "Professional",

    "Occ_Service_Labour":
        "Service / Labour",

    "Occ_Student":
        "Student"
}


TRAVEL_DUMMIES = {

    "TravelObj_Medical_Recreation":
        "Medical / Recreation",

    "TravelObj_Shopping":
        "Shopping",

    "TravelObj_Work":
        "Work"
}


occupation_sum = X[
    list(
        OCCUPATION_DUMMIES.keys()
    )
].sum(
    axis=1
)


travel_sum = X[
    list(
        TRAVEL_DUMMIES.keys()
    )
].sum(
    axis=1
)


assert (
    occupation_sum <= 1
).all(), (
    "Invalid occupation one-hot encoding."
)


assert (
    travel_sum <= 1
).all(), (
    "Invalid travel-objective one-hot encoding."
)


print(
    "One-hot structural validation: PASSED"
)


# ============================================================
# 14. DECODING MAPS
# ============================================================

GROUPED_MODE_MAP = {

    0:
        "Intermediate (Easybike/Rickshaw)",

    1:
        "Public (Bus)",

    2:
        "Low-cost Personal (Bike/Cycle)",

    3:
        "Private/Hired (CNG/Car)"
}


ORDINAL_MAP = {

    0:
        "No effect",

    1:
        "Low",

    2:
        "Medium",

    3:
        "High",

    4:
        "Extreme"
}


GENDER_MAP = {

    0:
        "Female",

    1:
        "Male"
}


BINARY_MAP = {

    0:
        "No",

    1:
        "Yes"
}


# ============================================================
# 15. VARIABLE GROUPS
# ============================================================

BINARY_FEATURES = {

    "Effect of Cost on Mode Choice",

    "Daily Activity Influence  Mode choice",

    "Distance Influence on Mode Choice",

    "Mobility Barrier on Mode Choice",

    "Accident Memories On Mode Choice"
}


ORDINAL_FEATURES = {

    "Importance of Cost in Mode Choice",

    "Importance of Safety in Mode Choice",

    "Environmental Impact on Mode Choice",

    "Trust level based on safety in Bus",

    "Trust level based on safety in CNG",

    "Trust level based on safety in Easybike",

    "Trust level based on safety in Rickshaw",

    "Trust level based on safety in Bike",

    "Trust level based on safety in Car",

    "Trust level based on safety in Cycle"
}


GROUPED_MODE_FEATURES = {

    "Mode_When_Depressed",

    "Mode_When_Happy",

    "Mode_When_Stressed",

    "Mode_When_Urgent",

    "Mode_In_Weather"
}


# ============================================================
# 16. FINAL ROLE MAPPING
#
# Economic      = 4
# Safety        = 10
# Psychological = 3
# Other         = 16
#
# These groups are predefined descriptive feature domains.
# ============================================================

ECONOMIC_FEATURES = {

    "Monthly Income",

    "Average Expendature for transportation",

    "Importance of Cost in Mode Choice",

    "Effect of Cost on Mode Choice"
}


SAFETY_FEATURES = {

    "Importance of Safety in Mode Choice",

    "Mobility Barrier on Mode Choice",

    "Accident Memories On Mode Choice",

    "Trust level based on safety in Bus",

    "Trust level based on safety in CNG",

    "Trust level based on safety in Easybike",

    "Trust level based on safety in Rickshaw",

    "Trust level based on safety in Bike",

    "Trust level based on safety in Car",

    "Trust level based on safety in Cycle"
}


PSYCHOLOGICAL_FEATURES = {

    "Mode_When_Depressed",

    "Mode_When_Happy",

    "Mode_When_Stressed"
}


OTHER_FEATURES = (

    set(
        feature_names
    )

    - ECONOMIC_FEATURES

    - SAFETY_FEATURES

    - PSYCHOLOGICAL_FEATURES
)


assert len(
    ECONOMIC_FEATURES
) == 4


assert len(
    SAFETY_FEATURES
) == 10


assert len(
    PSYCHOLOGICAL_FEATURES
) == 3


assert len(
    OTHER_FEATURES
) == 16


assert (
    len(
        ECONOMIC_FEATURES
    )
    +
    len(
        SAFETY_FEATURES
    )
    +
    len(
        PSYCHOLOGICAL_FEATURES
    )
    +
    len(
        OTHER_FEATURES
    )
) == 33


def get_role(
    feature
):

    if feature in ECONOMIC_FEATURES:

        return "Economic"


    if feature in SAFETY_FEATURES:

        return "Safety"


    if feature in PSYCHOLOGICAL_FEATURES:

        return "Psychological"


    return "Other"


print(
    "\nFinal descriptive role counts:"
)


print(
    "Economic      :",
    len(
        ECONOMIC_FEATURES
    )
)


print(
    "Safety        :",
    len(
        SAFETY_FEATURES
    )
)


print(
    "Psychological :",
    len(
        PSYCHOLOGICAL_FEATURES
    )
)


print(
    "Other         :",
    len(
        OTHER_FEATURES
    )
)


# ============================================================
# 17. CANONICAL FEATURE NAMES
# ============================================================

CANONICAL_NAMES = {

    "Gender":
        "gender",

    "Age":
        "age",

    "Monthly Income":
        "monthly_income",

    "Average Expendature for transportation":
        "transport_expenditure",

    "Importance of Cost in Mode Choice":
        "importance_cost",

    "Effect of Cost on Mode Choice":
        "cost_effect",

    "Daily Activity Influence  Mode choice":
        "daily_activity_influence",

    "Distance Influence on Mode Choice":
        "distance_influence",

    "Importance of Safety in Mode Choice":
        "importance_safety",

    "Mobility Barrier on Mode Choice":
        "mobility_barrier",

    "Accident Memories On Mode Choice":
        "accident_memories",

    "Environmental Impact on Mode Choice":
        "environmental_impact",

    "Trust level based on safety in Bus":
        "trust_bus",

    "Trust level based on safety in CNG":
        "trust_cng",

    "Trust level based on safety in Easybike":
        "trust_easybike",

    "Trust level based on safety in Rickshaw":
        "trust_rickshaw",

    "Trust level based on safety in Bike":
        "trust_bike",

    "Trust level based on safety in Car":
        "trust_car",

    "Trust level based on safety in Cycle":
        "trust_cycle",

    "Mode_When_Depressed":
        "mode_depressed",

    "Mode_When_Happy":
        "mode_happy",

    "Mode_When_Stressed":
        "mode_stressed",

    "Mode_When_Urgent":
        "mode_urgent",

    "Mode_In_Weather":
        "mode_weather",

    "Occ_Agriculture":
        "occupation_agriculture",

    "Occ_Business_SelfEmployed":
        "occupation_business_self_employed",

    "Occ_Homemaker":
        "occupation_homemaker",

    "Occ_Professional":
        "occupation_professional",

    "Occ_Service_Labour":
        "occupation_service_labour",

    "Occ_Student":
        "occupation_student",

    "TravelObj_Medical_Recreation":
        "travel_medical_recreation",

    "TravelObj_Shopping":
        "travel_shopping",

    "TravelObj_Work":
        "travel_work"
}


assert set(
    CANONICAL_NAMES.keys()
) == set(
    feature_names
)


# ============================================================
# 18. PYTHON SCALAR HELPERS
# ============================================================

def python_scalar(
    value
):

    if hasattr(
        value,
        "item"
    ):

        return value.item()


    return value


def nice_number(
    value
):

    value = python_scalar(
        value
    )


    try:

        value_float = float(
            value
        )


        if value_float.is_integer():

            return str(
                int(
                    value_float
                )
            )


        return (
            f"{value_float:.2f}"
        )


    except Exception:

        return str(
            value
        )


# ============================================================
# 19. RECONSTRUCT OCCUPATION
# ============================================================

def decode_occupation_profile(
    row
):

    active = [

        label

        for column, label
        in OCCUPATION_DUMMIES.items()

        if int(
            row[
                column
            ]
        ) == 1
    ]


    if len(active) == 0:

        return (
            "Unemployed "
            "(reference category)"
        )


    if len(active) == 1:

        return active[0]


    return (
        "Invalid multiple occupation "
        f"categories: {active}"
    )


# ============================================================
# 20. RECONSTRUCT TRAVEL OBJECTIVE
# ============================================================

def decode_travel_profile(
    row
):

    active = [

        label

        for column, label
        in TRAVEL_DUMMIES.items()

        if int(
            row[
                column
            ]
        ) == 1
    ]


    if len(active) == 0:

        return (
            "Education "
            "(reference category)"
        )


    if len(active) == 1:

        return active[0]


    return (
        "Invalid multiple travel "
        f"categories: {active}"
    )


# ============================================================
# 21. FEATURE VALUE DECODER
# ============================================================

def decode_value(
    feature,
    value
):

    value = python_scalar(
        value
    )


    # --------------------------------------------------------
    # GENDER
    # --------------------------------------------------------

    if feature == "Gender":

        try:

            decoded = GENDER_MAP.get(
                int(value),
                f"Gender code {value}"
            )

        except Exception:

            decoded = str(
                value
            )


        return {

            "raw":
                value,

            "decoded":
                decoded
        }


    # --------------------------------------------------------
    # BINARY
    # --------------------------------------------------------

    if feature in BINARY_FEATURES:

        try:

            decoded = BINARY_MAP.get(
                int(value),
                f"Binary code {value}"
            )

        except Exception:

            decoded = str(
                value
            )


        return {

            "raw":
                value,

            "decoded":
                decoded
        }


    # --------------------------------------------------------
    # ORDINAL
    # --------------------------------------------------------

    if feature in ORDINAL_FEATURES:

        try:

            decoded = ORDINAL_MAP.get(
                int(value),
                f"Ordinal code {value}"
            )

        except Exception:

            decoded = str(
                value
            )


        return {

            "raw":
                value,

            "decoded":
                decoded
        }


    # --------------------------------------------------------
    # CONTEXTUAL MODE VARIABLES
    #
    # CATEGORY IDENTIFIERS, NOT ORDINAL INTENSITY
    # --------------------------------------------------------

    if feature in GROUPED_MODE_FEATURES:

        try:

            decoded = GROUPED_MODE_MAP.get(
                int(value),
                f"Grouped mode code {value}"
            )

        except Exception:

            decoded = str(
                value
            )


        return {

            "raw":
                value,

            "decoded":
                decoded
        }


    # --------------------------------------------------------
    # AGE
    # --------------------------------------------------------

    if feature == "Age":

        return {

            "raw":
                value,

            "decoded":
                f"{nice_number(value)} years"
        }


    # --------------------------------------------------------
    # MONTHLY INCOME
    # --------------------------------------------------------

    if feature == "Monthly Income":

        return {

            "raw":
                value,

            "decoded":
                (
                    f"{nice_number(value)} "
                    "BDT/month"
                )
        }


    # --------------------------------------------------------
    # TRANSPORT EXPENDITURE
    # --------------------------------------------------------

    if (
        feature
        ==
        "Average Expendature for transportation"
    ):

        return {

            "raw":
                value,

            "decoded":
                (
                    f"{nice_number(value)} "
                    "BDT/month"
                )
        }


    # --------------------------------------------------------
    # OCCUPATION DUMMY
    # --------------------------------------------------------

    if feature in OCCUPATION_DUMMIES:

        category = (
            OCCUPATION_DUMMIES[
                feature
            ]
        )


        if int(value) == 1:

            decoded = (
                f"Yes — occupation category "
                f"is {category}"
            )

        else:

            decoded = (
                f"No — occupation category "
                f"is not {category}"
            )


        return {

            "raw":
                value,

            "decoded":
                decoded
        }


    # --------------------------------------------------------
    # TRAVEL OBJECTIVE DUMMY
    # --------------------------------------------------------

    if feature in TRAVEL_DUMMIES:

        category = (
            TRAVEL_DUMMIES[
                feature
            ]
        )


        if int(value) == 1:

            decoded = (
                f"Yes — travel objective "
                f"is {category}"
            )

        else:

            decoded = (
                f"No — travel objective "
                f"is not {category}"
            )


        return {

            "raw":
                value,

            "decoded":
                decoded
        }


    # --------------------------------------------------------
    # DEFAULT
    # --------------------------------------------------------

    return {

        "raw":
            value,

        "decoded":
            nice_number(
                value
            )
    }


# ============================================================
# 22. DESCRIPTIVE ROLE SUMMARY
#
# This is retained for audit/diagnostic purposes.
#
# It is NOT used to select the formal dominant role supplied
# to Qwen.
# ============================================================

def build_role_summary(
    all_records
):

    role_summary = {}


    for role in ROLE_CATEGORIES:

        role_records = [

            record

            for record
            in all_records

            if record[
                "role"
            ] == role
        ]


        positive_sum = sum(

            record[
                "shap_value"
            ]

            for record
            in role_records

            if record[
                "shap_value"
            ] > 0
        )


        negative_sum = sum(

            record[
                "shap_value"
            ]

            for record
            in role_records

            if record[
                "shap_value"
            ] < 0
        )


        net_sum = sum(

            record[
                "shap_value"
            ]

            for record
            in role_records
        )


        absolute_sum = sum(

            record[
                "absolute_shap"
            ]

            for record
            in role_records
        )


        role_summary[
            role
        ] = {

            "n_features":
                len(
                    role_records
                ),

            "positive_shap_sum":
                float(
                    positive_sum
                ),

            "negative_shap_sum":
                float(
                    negative_sum
                ),

            "net_shap":
                float(
                    net_sum
                ),

            "absolute_shap_sum":
                float(
                    absolute_sum
                )
        }


    return role_summary


# ============================================================
# 23. LOCAL TOP-2 ROLE SCORE HELPER
#
# IMPORTANT:
# ------------------------------------------------------------
# This does NOT calculate new SHAP values.
#
# It aggregates the already-fixed respondent-level
# predicted-class local SHAP values.
#
# For each role:
#
#   score =
#       largest |SHAP|
#       +
#       second-largest |SHAP|
#
# ============================================================

def get_role_top2(
    all_records,
    role
):

    role_records = [

        record

        for record
        in all_records

        if record[
            "role"
        ] == role
    ]


    assert len(
        role_records
    ) >= DOMINANT_ROLE_TOP_K, (
        f"Role {role} has fewer than "
        f"{DOMINANT_ROLE_TOP_K} features."
    )


    ranked_records = sorted(

        role_records,

        key=lambda record:
            record[
                "absolute_shap"
            ],

        reverse=True
    )


    top2_records = ranked_records[
        :DOMINANT_ROLE_TOP_K
    ]


    top2_abs_sum = sum(

        record[
            "absolute_shap"
        ]

        for record
        in top2_records
    )


    return (
        top2_records,
        float(
            top2_abs_sum
        )
    )


# ============================================================
# 24. PRE-COMPUTED LOCAL TOP-2 DOMINANT ROLE CONTEXT
#
# The role label comes from:
#
#     Dominant_Role_LocalTop2
#
# The role is NOT chosen here.
#
# This function:
#
#   1. retrieves the fixed supplied role;
#   2. extracts that role's exact Top-2 contributors;
#   3. calculates their absolute-SHAP sum;
#   4. internally verifies consistency with all 4 role scores.
#
# The LLM receives ONLY the supplied dominant role and
# its Top-2 contributors.
# ============================================================

def get_dominant_role_context(
    respondent_index,
    all_records
):

    respondent_index = int(
        respondent_index
    )


    row = dominant_role_df.loc[
        respondent_index
    ]


    supplied_role = str(
        row[
            "Dominant_Role_LocalTop2"
        ]
    ).strip()


    assert supplied_role in ROLE_CATEGORIES, (
        f"Invalid Local Top-2 dominant role "
        f"for respondent {respondent_index}: "
        f"{supplied_role}"
    )


    # --------------------------------------------------------
    # Calculate Local Top-2 scores from the already-existing
    # fixed SHAP values for INTERNAL CONSISTENCY CHECK ONLY.
    # --------------------------------------------------------

    role_top2_records = {}

    role_top2_scores = {}


    for role in ROLE_CATEGORIES:

        top2_records, top2_score = (
            get_role_top2(
                all_records,
                role
            )
        )


        role_top2_records[
            role
        ] = top2_records


        role_top2_scores[
            role
        ] = top2_score


    max_top2_score = max(
        role_top2_scores.values()
    )


    supplied_role_score = (
        role_top2_scores[
            supplied_role
        ]
    )


    # --------------------------------------------------------
    # Validation:
    #
    # The supplied dominant role must have a score equal to
    # the maximum Top-2 score.
    #
    # np.isclose() allows a genuine exact/near-exact tie
    # without arbitrarily changing the supplied role.
    # --------------------------------------------------------

    assert np.isclose(

        supplied_role_score,

        max_top2_score,

        rtol=1e-10,

        atol=1e-12

    ), (
        "Pre-computed dominant role does not match "
        "the fixed local Top-2 SHAP evidence for "
        f"respondent {respondent_index}. "
        f"Supplied role = {supplied_role}; "
        f"supplied-role score = {supplied_role_score:.12f}; "
        f"maximum score = {max_top2_score:.12f}; "
        f"all scores = {role_top2_scores}"
    )


    dominant_top2_records = (
        role_top2_records[
            supplied_role
        ]
    )


    # --------------------------------------------------------
    # Compact Top-2 evidence for LLM.
    #
    # Signed SHAP is retained.
    # Absolute SHAP is also supplied because role selection
    # is based on magnitude.
    # --------------------------------------------------------

    top2_contributors = []


    for rank, record in enumerate(
        dominant_top2_records,
        start=1
    ):

        top2_contributors.append({

            "rank":
                int(
                    rank
                ),

            "feature":
                record[
                    "feature"
                ],

            "canonical_feature":
                record[
                    "canonical_feature"
                ],

            "decoded_value":
                record[
                    "decoded_value"
                ],

            "signed_shap":
                round(
                    float(
                        record[
                            "shap_value"
                        ]
                    ),
                    6
                ),

            "absolute_shap":
                round(
                    float(
                        record[
                            "absolute_shap"
                        ]
                    ),
                    6
                ),

            "direction":
                record[
                    "direction"
                ]
        })


    return {

        "role":
            supplied_role,

        "selection_method":
            (
                "respondent-local Top-2 absolute "
                "predicted-class SHAP sum per "
                "predefined feature role"
            ),

        "top2_contributors":
            top2_contributors,

        "top2_absolute_shap_sum":
            round(
                float(
                    supplied_role_score
                ),
                6
            ),

        "interpretation":
            (
                f"{supplied_role} had the largest "
                "respondent-level Local Top-2 "
                "absolute SHAP sum among the four "
                "predefined roles."
            ),

        "source":
            (
                "Fixed pre-computed dominant-role label "
                "plus Top-2 contributors retrieved from "
                "the same fixed OOF predicted-class "
                "local SHAP evidence; not derived by "
                "the LLM."
            )
    }


print(
    "\nDominant-role method:"
)


print(
    "  respondent-local Top-2 absolute "
    "predicted-class SHAP sum per role"
)


print(
    "  LLM receives exact Top-2 contributors "
    "within the supplied dominant role."
)


# ============================================================
# 25. BUILD ONE RESPONDENT'S EVIDENCE
# ============================================================

def build_evidence(
    respondent_index
):

    row = X.iloc[
        respondent_index
    ]


    pred_row = pred_df.iloc[
        respondent_index
    ]


    shap_row = shap_df.iloc[
        respondent_index
    ]


    predicted_class = str(
        pred_row[
            "predicted_class"
        ]
    )


    # ========================================================
    # CLASS PROBABILITIES
    # ========================================================

    probability_dict = {

        class_name:
            float(
                pred_row[
                    PROBABILITY_COLUMNS[
                        class_name
                    ]
                ]
            )

        for class_name
        in classes
    }


    ranked_probabilities = sorted(

        probability_dict.items(),

        key=lambda item:
            item[1],

        reverse=True
    )


    predicted_probability = float(

        probability_dict[
            predicted_class
        ]
    )


    runner_up_class = (
        ranked_probabilities[
            1
        ][
            0
        ]
    )


    runner_up_probability = float(

        ranked_probabilities[
            1
        ][
            1
        ]
    )


    probability_margin = float(

        predicted_probability
        -
        runner_up_probability
    )


    assert np.isclose(

        predicted_probability,

        float(
            pred_row[
                "predicted_confidence"
            ]
        ),

        atol=1e-8

    ), (
        "Prediction-confidence mismatch "
        f"for respondent {respondent_index}"
    )


    # ========================================================
    # ALL 33 LOCAL SHAP FEATURE RECORDS
    # ========================================================

    all_records = []


    for feature in feature_names:

        shap_value = float(

            shap_row[
                SHAP_COLUMNS[
                    feature
                ]
            ]
        )


        decoded = decode_value(

            feature,

            row[
                feature
            ]
        )


        if shap_value > 0:

            direction = (
                "toward predicted class"
            )


        elif shap_value < 0:

            direction = (
                "away from predicted class"
            )


        else:

            direction = (
                "neutral"
            )


        all_records.append({

            "feature":
                feature,

            "canonical_feature":
                CANONICAL_NAMES[
                    feature
                ],

            "role":
                get_role(
                    feature
                ),

            "raw_value":
                decoded[
                    "raw"
                ],

            "decoded_value":
                decoded[
                    "decoded"
                ],

            "shap_value":
                shap_value,

            "absolute_shap":
                abs(
                    shap_value
                ),

            "direction":
                direction
        })


    # ========================================================
    # SUPPORTING EVIDENCE
    # ========================================================

    supporting = [

        record

        for record
        in all_records

        if (
            record[
                "shap_value"
            ] > 0

            and

            record[
                "absolute_shap"
            ] >= MIN_ABS_SHAP
        )
    ]


    supporting = sorted(

        supporting,

        key=lambda record:
            record[
                "shap_value"
            ],

        reverse=True
    )


    supporting = supporting[
        :TOP_POSITIVE
    ]


    # ========================================================
    # OPPOSING EVIDENCE
    # ========================================================

    opposing = [

        record

        for record
        in all_records

        if (
            record[
                "shap_value"
            ] < 0

            and

            record[
                "absolute_shap"
            ] >= MIN_ABS_SHAP
        )
    ]


    opposing = sorted(

        opposing,

        key=lambda record:
            record[
                "shap_value"
            ]
    )


    opposing = opposing[
        :TOP_NEGATIVE
    ]


    # ========================================================
    # RESPONDENT PROFILE CONTEXT
    # ========================================================

    profile_context = {

        "gender":
            decode_value(
                "Gender",
                row[
                    "Gender"
                ]
            )[
                "decoded"
            ],

        "age":
            decode_value(
                "Age",
                row[
                    "Age"
                ]
            )[
                "decoded"
            ],

        "occupation_group":
            decode_occupation_profile(
                row
            ),

        "travel_objective":
            decode_travel_profile(
                row
            )
    }


    # ========================================================
    # DESCRIPTIVE ROLE SUMMARY
    #
    # Retained internally for diagnostics.
    # NOT the formal dominant-role method.
    # ========================================================

    role_summary = build_role_summary(
        all_records
    )


    # ========================================================
    # FIXED RESPONDENT-LOCAL TOP-2 DOMINANT ROLE
    #
    # Includes:
    #
    #   - fixed supplied role
    #   - exact Top-2 features
    #   - signed SHAP values
    #   - absolute SHAP values
    #   - Local Top-2 absolute SHAP sum
    # ========================================================

    dominant_role_context = (
        get_dominant_role_context(
            respondent_index,
            all_records
        )
    )


    # ========================================================
    # FINAL LLM OBJECT
    #
    # true_class / correct intentionally excluded.
    # ========================================================

    output = {

        "respondent_id":
            int(
                respondent_index
            ),

        "respondent_number":
            int(
                respondent_index
                + 1
            ),


        "model_source": {

            "classifier":
                "Random Forest",

            "resampling":
                (
                    "RandomOverSampler "
                    "on training folds only"
                ),

            "validation":
                (
                    "fixed stratified "
                    "5-fold out-of-fold"
                ),

            "random_state":
                42,

            "feature_count":
                33
        },


        "model_output": {

            "predicted_mode":
                predicted_class,

            "predicted_probability":
                predicted_probability,

            "runner_up_mode":
                runner_up_class,

            "runner_up_probability":
                runner_up_probability,

            "probability_margin":
                probability_margin,

            "all_class_probabilities":
                probability_dict
        },


        "respondent_profile_context":
            profile_context,


        "supporting_evidence":
            supporting,


        "opposing_evidence":
            opposing,


        "dominant_role":
            dominant_role_context,


        # Internal descriptive context.
        # Cell E should NOT send this to Qwen.
        "role_summary":
            role_summary,


        "role_summary_note":
            (
                "Role-level SHAP totals are diagnostic "
                "context only. The formal respondent-level "
                "dominant role is the separately supplied "
                "Local Top-2 absolute-SHAP classification."
            ),


        "llm_instructions": {

            "task":
                (
                    "Explain the fixed Random Forest "
                    "transport-mode prediction using only "
                    "the supplied OOF model probabilities, "
                    "predicted-class local SHAP evidence, "
                    "and pre-computed dominant-role evidence."
                ),


            "rules": [

                (
                    "Do not make a new transport-mode "
                    "prediction."
                ),

                (
                    "Do not change or challenge the "
                    "Random Forest predicted mode."
                ),

                (
                    "Do not claim that the respondent "
                    "actually chose the predicted mode."
                ),

                (
                    "Use only the supplied model evidence."
                ),

                (
                    "Do not introduce an unsupplied feature, "
                    "value, probability, preference, "
                    "motivation, or behavioural explanation."
                ),

                (
                    "Positive SHAP values are model evidence "
                    "toward the predicted class."
                ),

                (
                    "Negative SHAP values are model evidence "
                    "away from the predicted class."
                ),

                (
                    "The dominant-role label is fixed and "
                    "pre-computed outside the LLM."
                ),

                (
                    "Do not independently calculate, infer, "
                    "modify, or replace the dominant role."
                ),

                (
                    "Report the two supplied Top-2 "
                    "contributors within the dominant role."
                ),

                (
                    "Report their signed SHAP values exactly "
                    "as supplied."
                ),

                (
                    "Report the supplied dominant-role "
                    "Top-2 absolute SHAP sum exactly."
                ),

                (
                    "The dominant role means only that this "
                    "role had the largest respondent-level "
                    "Local Top-2 absolute SHAP sum among "
                    "the four predefined roles."
                ),

                (
                    "Do not translate dominant role into "
                    "'strongest influence', 'most important', "
                    "'most significant', 'main driver', or "
                    "other broader claims."
                ),

                (
                    "A dominant-role Top-2 contributor may "
                    "have either a positive or negative "
                    "signed SHAP value because role selection "
                    "uses absolute SHAP magnitude."
                ),

                (
                    "Mode_When_Depressed, Mode_When_Happy, "
                    "Mode_When_Stressed, Mode_When_Urgent, "
                    "and Mode_In_Weather are categorical "
                    "transport-group identifiers, not "
                    "ordinal scales or direct statements "
                    "of behavioural preference."
                ),

                (
                    "Occupation and travel-objective "
                    "variables are one-hot indicators."
                ),

                (
                    "Do not use causal language such as "
                    "'caused', 'made the respondent choose', "
                    "'determined', or 'resulted in'."
                ),

                (
                    "SHAP explains fitted-model prediction "
                    "behaviour, not causal transport-choice "
                    "behaviour."
                )
            ],


            "requested_output_format": {

                "predicted_mode":
                    (
                        "state the supplied Random Forest "
                        "predicted mode"
                    ),

                "prediction_probability":
                    (
                        "state the supplied predicted-class "
                        "probability"
                    ),

                "runner_up_mode":
                    (
                        "state the supplied second-highest "
                        "probability mode"
                    ),

                "runner_up_probability":
                    (
                        "state its supplied probability"
                    ),

                "probability_margin":
                    (
                        "state the supplied difference "
                        "between the top two probabilities"
                    ),

                "supporting_evidence":
                    (
                        "report the supplied positive "
                        "SHAP evidence"
                    ),

                "opposing_evidence":
                    (
                        "report the supplied negative "
                        "SHAP evidence"
                    ),

                "uncertainty_note":
                    (
                        "describe only the supplied numerical "
                        "probability separation"
                    ),

                "dominant_role":
                    (
                        "report the supplied Local Top-2 "
                        "dominant role exactly"
                    ),

                "dominant_role_top2_contributors":
                    (
                        "report the two supplied features "
                        "forming the dominant-role Top-2 score "
                        "with their signed SHAP values"
                    ),

                "dominant_role_top2_absolute_shap_sum":
                    (
                        "report the supplied Local Top-2 "
                        "absolute SHAP sum exactly"
                    ),

                "overall_explanation":
                    (
                        "provide a concise model-grounded "
                        "synthesis without causal or "
                        "behavioural over-interpretation"
                    )
            }
        }
    }


    return output


# ============================================================
# 26. BUILD ALL 348 LLM-READY OBJECTS
# ============================================================

all_json = []


for respondent_index in range(
    len(X)
):

    evidence = build_evidence(
        respondent_index
    )


    all_json.append(
        evidence
    )


assert len(
    all_json
) == len(
    X
)


print(
    "\nLLM-ready respondents:",
    len(
        all_json
    )
)


# ============================================================
# 27. SAVE FULL JSON
# ============================================================

JSON_PATH = os.path.join(

    OUTPUT_DIR,

    "llm_ready_ros_rf_final.json"
)


with open(

    JSON_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        all_json,

        f,

        indent=2,

        ensure_ascii=False
    )


# ============================================================
# 28. SAVE JSONL
# ============================================================

JSONL_PATH = os.path.join(

    OUTPUT_DIR,

    "llm_ready_ros_rf_final.jsonl"
)


with open(

    JSONL_PATH,

    "w",

    encoding="utf-8"

) as f:

    for item in all_json:

        f.write(

            json.dumps(

                item,

                ensure_ascii=False
            )

            + "\n"
        )


# ============================================================
# 29. AUDIT MASTER CSV
#
# True class / correctness remain audit-only.
# ============================================================

audit_rows = []


for respondent_index, item in enumerate(
    all_json
):

    model_output = item[
        "model_output"
    ]


    dominant = item[
        "dominant_role"
    ]


    dominant_top2 = dominant[
        "top2_contributors"
    ]


    assert len(
        dominant_top2
    ) == 2


    supporting_text = " | ".join([

        (
            f"{record['feature']} "
            f"[{record['role']}] "
            f"= {record['decoded_value']} "
            f"(SHAP {record['shap_value']:+.6f})"
        )

        for record
        in item[
            "supporting_evidence"
        ]
    ])


    opposing_text = " | ".join([

        (
            f"{record['feature']} "
            f"[{record['role']}] "
            f"= {record['decoded_value']} "
            f"(SHAP {record['shap_value']:+.6f})"
        )

        for record
        in item[
            "opposing_evidence"
        ]
    ])


    audit_rows.append({

        "respondent_id":
            respondent_index,

        "respondent_number":
            respondent_index + 1,

        # AUDIT ONLY
        "true_class":
            pred_df.iloc[
                respondent_index
            ][
                "true_class"
            ],

        # AUDIT ONLY
        "correct":
            int(
                pred_df.iloc[
                    respondent_index
                ][
                    "correct"
                ]
            ),

        "predicted_mode":
            model_output[
                "predicted_mode"
            ],

        "predicted_probability":
            model_output[
                "predicted_probability"
            ],

        "runner_up_mode":
            model_output[
                "runner_up_mode"
            ],

        "runner_up_probability":
            model_output[
                "runner_up_probability"
            ],

        "probability_margin":
            model_output[
                "probability_margin"
            ],

        "occupation_group":
            item[
                "respondent_profile_context"
            ][
                "occupation_group"
            ],

        "travel_objective":
            item[
                "respondent_profile_context"
            ][
                "travel_objective"
            ],

        "supporting_evidence":
            supporting_text,

        "opposing_evidence":
            opposing_text,

        # ----------------------------------------------------
        # FINAL DOMINANT-ROLE AUDIT FIELDS
        # ----------------------------------------------------

        "dominant_role":
            dominant[
                "role"
            ],

        "dominant_top1_feature":
            dominant_top2[
                0
            ][
                "feature"
            ],

        "dominant_top1_signed_shap":
            dominant_top2[
                0
            ][
                "signed_shap"
            ],

        "dominant_top1_absolute_shap":
            dominant_top2[
                0
            ][
                "absolute_shap"
            ],

        "dominant_top2_feature":
            dominant_top2[
                1
            ][
                "feature"
            ],

        "dominant_top2_signed_shap":
            dominant_top2[
                1
            ][
                "signed_shap"
            ],

        "dominant_top2_absolute_shap":
            dominant_top2[
                1
            ][
                "absolute_shap"
            ],

        "dominant_role_top2_absolute_shap_sum":
            dominant[
                "top2_absolute_shap_sum"
            ],

        # ----------------------------------------------------
        # DESCRIPTIVE INTERNAL ROLE DIAGNOSTICS
        # ----------------------------------------------------

        "economic_net_shap":
            item[
                "role_summary"
            ][
                "Economic"
            ][
                "net_shap"
            ],

        "safety_net_shap":
            item[
                "role_summary"
            ][
                "Safety"
            ][
                "net_shap"
            ],

        "psychological_net_shap":
            item[
                "role_summary"
            ][
                "Psychological"
            ][
                "net_shap"
            ],

        "other_net_shap":
            item[
                "role_summary"
            ][
                "Other"
            ][
                "net_shap"
            ]
    })


audit_df = pd.DataFrame(
    audit_rows
)


AUDIT_PATH = os.path.join(

    OUTPUT_DIR,

    "agentic_ai_audit_master_ros_rf.csv"
)


audit_df.to_csv(

    AUDIT_PATH,

    index=False
)


# ============================================================
# 30. COMPACT LLM REVIEW CSV
#
# Removes true outcome and correctness.
# ============================================================

review_df = audit_df.drop(

    columns=[
        "true_class",
        "correct"
    ]

).copy()


REVIEW_PATH = os.path.join(

    OUTPUT_DIR,

    "llm_input_manual_review_ros_rf.csv"
)


review_df.to_csv(

    REVIEW_PATH,

    index=False
)


# ============================================================
# 31. FEATURE MAPPING DIAGNOSTIC
# ============================================================

feature_mapping_rows = []


for feature in feature_names:

    feature_mapping_rows.append({

        "feature":
            feature,

        "canonical_feature":
            CANONICAL_NAMES[
                feature
            ],

        "role":
            get_role(
                feature
            ),

        "shap_column":
            SHAP_COLUMNS[
                feature
            ]
    })


feature_mapping_df = pd.DataFrame(
    feature_mapping_rows
)


FEATURE_MAPPING_PATH = os.path.join(

    OUTPUT_DIR,

    "agentic_feature_mapping_ros_rf.csv"
)


feature_mapping_df.to_csv(

    FEATURE_MAPPING_PATH,

    index=False
)


# ============================================================
# 32. BASIC SUMMARY
# ============================================================

print(
    "\n" + "=" * 90
)


print(
    "AGENTIC-AI INPUT SUMMARY"
)


print(
    "=" * 90
)


print(
    "Total respondents:",
    len(
        all_json
    )
)


print(
    "\nPredicted class counts:"
)


print(

    audit_df[
        "predicted_mode"
    ]

    .value_counts()
)


print(
    "\nDominant role counts "
    "(LOCAL TOP-2 ABSOLUTE SHAP):"
)


print(

    audit_df[
        "dominant_role"
    ]

    .value_counts()
)


print(
    "\nCorrect / incorrect "
    "(AUDIT ONLY):"
)


print(

    audit_df[
        "correct"
    ]

    .value_counts()
)


print(
    "\nProbability-margin summary:"
)


print(

    audit_df[
        "probability_margin"
    ]

    .describe()
)


print(
    "\nDominant-role Top-2 absolute SHAP sum summary:"
)


print(

    audit_df[
        "dominant_role_top2_absolute_shap_sum"
    ]

    .describe()
)


# ============================================================
# 33. PRINT RESPONDENT 0
# ============================================================

print(
    "\n" + "=" * 90
)


print(
    "RESPONDENT 0 — FINAL LLM INPUT"
)


print(
    "=" * 90
)


print(

    json.dumps(

        all_json[
            0
        ],

        indent=2,

        ensure_ascii=False

    )[:20000]
)


# ============================================================
# 34. FINAL VALIDATION
# ============================================================

for item in all_json:

    # True outcome must never enter LLM payload.
    assert (
        "true_class"
        not in item
    )


    assert (
        "correct"
        not in item
    )


    assert (
        item[
            "model_output"
        ][
            "predicted_mode"
        ]
        in classes
    )


    assert (
        0
        <=
        item[
            "model_output"
        ][
            "predicted_probability"
        ]
        <=
        1
    )


    dominant = item[
        "dominant_role"
    ]


    assert (
        dominant[
            "role"
        ]
        in ROLE_CATEGORIES
    )


    assert len(
        dominant[
            "top2_contributors"
        ]
    ) == 2


    assert (
        dominant[
            "top2_absolute_shap_sum"
        ]
        >= 0
    )


    # Verify sum after 6-decimal representation.
    top2_rounded_sum = round(

        sum(

            contributor[
                "absolute_shap"
            ]

            for contributor
            in dominant[
                "top2_contributors"
            ]
        ),

        6
    )


    # Small tolerance because the overall score is rounded
    # from full-precision SHAP values, whereas the contributor
    # values supplied to the LLM are stored to 6 decimals.
    assert np.isclose(

        top2_rounded_sum,

        dominant[
            "top2_absolute_shap_sum"
        ],

        atol=2e-6

    ), (
        "Dominant-role Top-2 sum mismatch "
        f"for respondent {item['respondent_id']}"
    )


print(
    "\nFinal LLM-payload validation: PASSED"
)


print(
    "Dominant-role Top-2 contributor validation: PASSED"
)


# ============================================================
# 35. FINISHED
# ============================================================

print(
    "\n" + "=" * 90
)


print(
    "FINAL ROS + RF AGENTIC-AI DATASET CREATED"
)


print(
    "=" * 90
)


print(
    "\nNO MODEL RETRAINING WAS PERFORMED."
)


print(
    "NO NEW ROS WAS PERFORMED."
)


print(
    "NO NEW SHAP VALUES WERE CALCULATED."
)


print(
    "\nDominant role:"
)


print(
    "Pre-computed respondent-local Top-2 "
    "absolute predicted-class SHAP role."
)


print(
    "\nDominant-role explanation evidence:"
)


print(
    "Exact two highest-|SHAP| features within "
    "the supplied dominant role + signed SHAP "
    "values + Top-2 absolute SHAP sum."
)


print(
    "\nSaved files:"
)


print(
    "1.",
    JSON_PATH
)


print(
    "2.",
    JSONL_PATH
)


print(
    "3.",
    AUDIT_PATH
)


print(
    "4.",
    REVIEW_PATH
)


print(
    "5.",
    FEATURE_MAPPING_PATH
)


FINAL AGENTIC-AI EVIDENCE BUILDER — ROS + RF + OOF SHAP + LOCAL TOP-2 DOMINANT ROLE
Dataset rows       : 348
Prediction rows    : 348
Local SHAP rows    : 348
Dominant-role rows : 348

Final predictors: 33
01. Gender
02. Age
03. Monthly Income
04. Average Expendature for transportation
05. Importance of Cost in Mode Choice
06. Effect of Cost on Mode Choice
07. Daily Activity Influence  Mode choice
08. Distance Influence on Mode Choice
09. Importance of Safety in Mode Choice
10. Mobility Barrier on Mode Choice
11. Accident Memories On Mode Choice
12. Environmental Impact on Mode Choice
13. Trust level based on safety in Bus
14. Trust level based on safety in CNG
15. Trust level based on safety in Easybike
16. Trust level based on safety in Rickshaw
17. Trust level based on safety in Bike
18. Trust level based on safety in Car
19. Trust level based on safety in Cycle
20. Mode_When_Depressed
21. Mode_When_Happy
22. Mode_When_Stressed
23. Mode_When_Urgent
24. Mode_In_Weather
25. Occ_Agricu

## Load the fixed LLM evidence payload

In [ ]:
# ============================================================
# CELL C — LOAD FINAL FIXED RF + OOF SHAP EVIDENCE
# ============================================================

import os
import json


INPUT_JSON = (
    "/content/drive/MyDrive/Research/Transport Mode/"
    "FINAL TRANSPORT MODE/Q1 PIPELINE V2/Agentic AI/"
    "llm_ready_ros_rf_final.json"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/Research/Transport Mode/"
    "FINAL TRANSPORT MODE/Q1 PIPELINE V2/Qwen3 Agentic AI Explanations"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


assert os.path.exists(INPUT_JSON), (
    f"Input JSON not found:\n{INPUT_JSON}"
)


with open(
    INPUT_JSON,
    "r",
    encoding="utf-8"
) as f:

    evidence_data = json.load(f)


assert len(evidence_data) == 348, (
    f"Expected 348 respondents, found {len(evidence_data)}."
)


for item in evidence_data:

    assert "respondent_id" in item

    assert "model_output" in item

    assert "supporting_evidence" in item

    assert "opposing_evidence" in item

    # True outcome must never enter the LLM payload.
    assert "true_class" not in item

    assert "correct" not in item


print("=" * 80)
print("FINAL LLM EVIDENCE PAYLOAD")
print("=" * 80)

print("Input JSON       :", INPUT_JSON)
print("Output directory :", OUTPUT_DIR)
print("Respondents      :", len(evidence_data))
print("Payload validation: PASSED")


FINAL LLM EVIDENCE PAYLOAD
Input JSON       : /content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/Agentic AI/llm_ready_ros_rf_final.json
Output directory : /content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/Qwen3 Agentic AI Explanations
Respondents      : 348
Payload validation: PASSED


## Load Qwen3-8B in 4-bit mode

In [ ]:
# ============================================================
# CELL D — QWEN3-8B 4-BIT LOAD / RECOVERY
# ============================================================

import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


# Release stale inactive CUDA blocks before checking/reusing the model.
gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


MODEL_NAME = "Qwen/Qwen3-8B"


def _existing_qwen_is_valid():

    if "model" not in globals():

        return False

    if "tokenizer" not in globals():

        return False

    try:

        model_name = str(
            getattr(
                model.config,
                "_name_or_path",
                ""
            )
        )

        if MODEL_NAME.lower() not in model_name.lower():

            return False

        if not getattr(
            model,
            "is_loaded_in_4bit",
            False
        ):

            return False

        if next(
            model.parameters()
        ).device.type != "cuda":

            return False

        cpu_params = [
            name
            for name, parameter
            in model.named_parameters()
            if parameter.device.type == "cpu"
        ]

        if cpu_params:

            return False

        return True

    except Exception:

        return False


if _existing_qwen_is_valid():

    print("Valid Qwen3-8B 4-bit model already loaded. Reusing it.")

else:

    # Remove incompatible/stale variables if present.
    if "model" in globals():

        try:
            del model
        except Exception:
            pass

    if "tokenizer" in globals():

        try:
            del tokenizer
        except Exception:
            pass

    gc.collect()

    torch.cuda.empty_cache()


    print("=" * 80)
    print("LOADING QWEN3-8B IN 4-BIT MODE")
    print("=" * 80)

    print(
        "GPU allocated before load:",
        round(
            torch.cuda.memory_allocated() / 1024**3,
            2
        ),
        "GB"
    )


    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True
    )


    quantization_config = BitsAndBytesConfig(

        load_in_4bit=True,

        bnb_4bit_quant_type="nf4",

        bnb_4bit_use_double_quant=True,

        bnb_4bit_compute_dtype=torch.float16
    )


    model = AutoModelForCausalLM.from_pretrained(

        MODEL_NAME,

        quantization_config=quantization_config,

        dtype=torch.float16,

        device_map={"": 0},

        trust_remote_code=True
    )


    model.eval()


# ============================================================
# STRICT MODEL VALIDATION
# ============================================================

cpu_parameters = [

    name

    for name, parameter
    in model.named_parameters()

    if parameter.device.type == "cpu"
]


assert getattr(
    model,
    "is_loaded_in_4bit",
    False
), (
    "Qwen3-8B was not loaded in 4-bit mode."
)


assert next(
    model.parameters()
).device.type == "cuda", (
    "Qwen3-8B is not on the CUDA GPU."
)


assert len(
    cpu_parameters
) == 0, (
    "Some Qwen model parameters are offloaded to CPU."
)


print("\n" + "=" * 80)
print("QWEN3-8B MODEL VALIDATION")
print("=" * 80)

print(
    "Model name:",
    getattr(
        model.config,
        "_name_or_path",
        "unknown"
    )
)

print(
    "Tokenizer:",
    type(tokenizer).__name__
)

print(
    "Loaded in 4-bit:",
    getattr(
        model,
        "is_loaded_in_4bit",
        False
    )
)

print(
    "First parameter device:",
    next(
        model.parameters()
    ).device
)

print(
    "Model memory footprint:",
    round(
        model.get_memory_footprint() / 1024**3,
        2
    ),
    "GB"
)

print(
    "CPU parameter count:",
    len(
        cpu_parameters
    )
)

print(
    "GPU allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print(
    "GPU reserved:",
    round(
        torch.cuda.memory_reserved() / 1024**3,
        2
    ),
    "GB"
)

print("\nQWEN3-8B 4-BIT GPU LOAD: PASSED")


LOADING QWEN3-8B IN 4-BIT MODE
GPU allocated before load: 0.0 GB


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


QWEN3-8B MODEL VALIDATION
Model name: Qwen/Qwen3-8B
Tokenizer: Qwen2Tokenizer
Loaded in 4-bit: True
First parameter device: cuda:0
Model memory footprint: 5.55 GB
CPU parameter count: 0
GPU allocated: 5.67 GB
GPU reserved: 5.76 GB

QWEN3-8B 4-BIT GPU LOAD: PASSED


## Constrained evidence-grounded prompt

In [ ]:
# ============================================================
# CELL E — FINAL MEMORY-OPTIMIZED SHAP-GROUNDED QWEN PROMPT
# STRICT LOCAL TOP-2 DOMINANT-ROLE EXPLANATION
# ============================================================

import json


# ============================================================
# SYSTEM PROMPT
#
# Keep deliberately compact.
# Detailed fidelity is enforced again by Cell F audit.
# ============================================================

SYSTEM_PROMPT = """
You are a constrained explainable-AI reporting agent.

Explain only the fixed Random Forest prediction supplied in
MODEL EVIDENCE.

Do not make a new prediction.

Do not infer the respondent's actual transport choice.


RULES

1. Use only MODEL EVIDENCE.

Positive SHAP = model evidence toward the predicted class.

Negative SHAP = model evidence away from the predicted class.

SHAP describes fitted-model prediction behaviour, not causation.


2. Report every supplied supporting and opposing item with:

- exact feature name
- exact decoded value
- exact signed SHAP
- exact direction

Do not paraphrase decoded values.

Do not convert numeric values into qualitative labels.


3. The following are categorical transport-group identifiers:

- Mode_When_Depressed
- Mode_When_Happy
- Mode_When_Stressed
- Mode_When_Urgent
- Mode_In_Weather

Never interpret these as:

- preference
- tendency
- motivation
- emotional intensity
- behaviour
- actual choice

Report only the supplied decoded category and SHAP evidence.


4. Occupation and travel-objective variables are one-hot
indicators.

Report only their supplied decoded meanings.

Do not infer an alternative occupation or travel purpose.


5. The dominant role is pre-computed outside the LLM.

For each predefined role:

- Economic
- Safety
- Psychological
- Other

the two largest absolute respondent-level predicted-class
local SHAP values were summed.

The role with the largest Local Top-2 absolute SHAP sum was
selected.

Do not recalculate, infer, modify, challenge, or replace the
supplied dominant role.


6. "Dominant role" means ONLY:

the role had the largest respondent-level Local Top-2
absolute SHAP sum among the four predefined roles.

Never describe the dominant role as:

- strongest influence
- strongest local influence
- strongest evidence
- strongest local evidence
- most influential
- most important
- most significant
- main driver
- primary driver
- causal determinant


7. Report exactly the two supplied dominant-role Top-2
contributors.

For each report:

- rank
- exact feature name
- exact decoded value
- exact signed SHAP
- exact absolute SHAP
- exact direction

A Top-2 contributor may have a negative signed SHAP because
dominant-role selection uses absolute SHAP magnitude.


8. Report the supplied dominant-role Top-2 sum exactly as the
supplied text representation.

Do not recalculate or round it.


9. Copy the supplied dominant-role interpretation sentence
exactly.

Do not paraphrase it.


10. Do not use causal, behavioural, preference, or unsupplied
magnitude wording.

Do not use terms such as:

- caused
- determined
- preference
- strongest
- significant
- notable
- major
- minor
- smaller
- larger
- slightly
- mainly
- primarily
- substantially

Do not convert:

- Extreme -> high
- Medium -> moderate
- Low -> weak
- numeric income -> high/low income
- numeric expenditure -> high/low/moderate expenditure


11. The Uncertainty note must report only:

- predicted probability
- runner-up probability
- probability margin

Do not add qualitative confidence or magnitude labels.


12. If no opposing evidence is supplied, write exactly:

"No opposing feature passed the supplied evidence inclusion threshold."


13. OVERALL EXPLANATION

The Overall explanation must contain EXACTLY TWO sentences.

Sentence 1 must state only that:

- supplied positive SHAP evidence provided evidence toward
  the fixed Random Forest predicted class; and
- supplied negative SHAP evidence, when present, provided
  evidence away from that predicted class.

Sentence 2 must be the supplied dominant-role interpretation
sentence EXACTLY.

Do NOT repeat the Top-2 contributor details in the Overall
explanation. They are already reported in the dedicated
Top-2 section.

Do not independently regroup general features by role.

Do not add any additional interpretation.


14. Do not provide policy, intervention, transport-planning,
or behavioural recommendations.


RETURN EXACTLY THIS STRUCTURE:


Predicted mode:
[exact supplied predicted mode]

Prediction probability:
[exact supplied value]

Runner-up mode:
[exact supplied runner-up mode]

Runner-up probability:
[exact supplied value]

Probability margin:
[exact supplied value]

Supporting evidence:
- Feature: [exact feature] | Value: [exact decoded value] | SHAP: [exact signed SHAP] | Direction: [exact direction]

Opposing evidence:
- Feature: [exact feature] | Value: [exact decoded value] | SHAP: [exact signed SHAP] | Direction: [exact direction]

Uncertainty note:
[only the three supplied probability quantities]

Dominant role:
[exact supplied role]

Top-2 contributors within dominant role:
- Rank 1 | Feature: [exact feature] | Value: [exact decoded value] | SHAP: [exact signed SHAP] | |SHAP|: [exact absolute SHAP] | Direction: [exact direction]
- Rank 2 | Feature: [exact feature] | Value: [exact decoded value] | SHAP: [exact signed SHAP] | |SHAP|: [exact absolute SHAP] | Direction: [exact direction]

Dominant-role Top-2 absolute SHAP sum:
[exact supplied sum text]

Dominant-role interpretation:
[copy supplied interpretation exactly]

Overall explanation:
[EXACTLY two sentences. Sentence 1 = SHAP direction summary only.
Sentence 2 = supplied dominant-role interpretation exactly.]
"""


# ============================================================
# COMPACT GENERAL FEATURE
# ============================================================

def compact_feature(record):

    shap_value = float(
        record[
            "shap_value"
        ]
    )

    return {

        "feature":
            str(
                record[
                    "feature"
                ]
            ),

        "value":
            str(
                record[
                    "decoded_value"
                ]
            ),

        "shap_text":
            f"{shap_value:+.6f}",

        "direction":
            str(
                record[
                    "direction"
                ]
            ),
    }


# ============================================================
# COMPACT DOMINANT-ROLE CONTRIBUTOR
# ============================================================

def compact_dominant_contributor(record):

    signed_shap = float(
        record[
            "signed_shap"
        ]
    )

    absolute_shap = float(
        record[
            "absolute_shap"
        ]
    )

    return {

        "rank":
            int(
                record[
                    "rank"
                ]
            ),

        "feature":
            str(
                record[
                    "feature"
                ]
            ),

        "value":
            str(
                record[
                    "decoded_value"
                ]
            ),

        "signed_shap_text":
            f"{signed_shap:+.6f}",

        "absolute_shap_text":
            f"{absolute_shap:.6f}",

        "direction":
            str(
                record[
                    "direction"
                ]
            ),
    }


# ============================================================
# COMPACT DOMINANT ROLE
# ============================================================

def compact_dominant_role(
    dominant_role
):

    assert dominant_role is not None, (
        "dominant_role is missing."
    )

    contributors = (
        dominant_role[
            "top2_contributors"
        ]
    )

    assert len(
        contributors
    ) == 2, (
        "Exactly two dominant-role contributors are required."
    )

    top2_sum = float(
        dominant_role[
            "top2_absolute_shap_sum"
        ]
    )

    return {

        "role":
            str(
                dominant_role[
                    "role"
                ]
            ).strip(),

        "top2_contributors": [

            compact_dominant_contributor(
                record
            )

            for record
            in contributors
        ],

        "top2_sum_text":
            f"{top2_sum:.6f}",

        "interpretation":
            str(
                dominant_role[
                    "interpretation"
                ]
            ).strip(),
    }


# ============================================================
# BUILD MEMORY-OPTIMIZED USER PROMPT
#
# Short labelled lines reduce prompt tokens / GPU attention
# memory compared with verbose JSON.
# ============================================================

def build_user_prompt(
    item
):

    model_output = (
        item[
            "model_output"
        ]
    )

    supporting = [

        compact_feature(
            record
        )

        for record
        in item.get(
            "supporting_evidence",
            []
        )
    ]

    opposing = [

        compact_feature(
            record
        )

        for record
        in item.get(
            "opposing_evidence",
            []
        )
    ]

    dominant = (
        compact_dominant_role(
            item[
                "dominant_role"
            ]
        )
    )


    lines = [

        "MODEL EVIDENCE — REPORT ONLY; DO NOT INFER",

        f"Respondent ID: "
        f"{int(item['respondent_id'])}",

        f"Predicted mode: "
        f"{model_output['predicted_mode']}",

        f"Prediction probability: "
        f"{float(model_output['predicted_probability']):g}",

        f"Runner-up mode: "
        f"{model_output['runner_up_mode']}",

        f"Runner-up probability: "
        f"{float(model_output['runner_up_probability']):g}",

        f"Probability margin: "
        f"{float(model_output['probability_margin']):g}",

        "",

        "SUPPORTING EVIDENCE "
        "(report every line exactly):",
    ]


    # ========================================================
    # SUPPORTING EVIDENCE
    # ========================================================

    if supporting:

        for idx, record in enumerate(
            supporting,
            start=1
        ):

            lines.append(

                f"{idx}. "
                f"Feature={record['feature']} | "
                f"Value={record['value']} | "
                f"SHAP={record['shap_text']} | "
                f"Direction={record['direction']}"
            )

    else:

        lines.append(
            "None supplied."
        )


    # ========================================================
    # OPPOSING EVIDENCE
    # ========================================================

    lines.extend([

        "",

        "OPPOSING EVIDENCE "
        "(report every line exactly):",
    ])


    if opposing:

        for idx, record in enumerate(
            opposing,
            start=1
        ):

            lines.append(

                f"{idx}. "
                f"Feature={record['feature']} | "
                f"Value={record['value']} | "
                f"SHAP={record['shap_text']} | "
                f"Direction={record['direction']}"
            )

    else:

        lines.append(

            "No opposing feature passed the supplied "
            "evidence inclusion threshold."
        )


    # ========================================================
    # DOMINANT ROLE
    # ========================================================

    lines.extend([

        "",

        "PRE-COMPUTED DOMINANT ROLE "
        "(do not recalculate):",

        f"Role: "
        f"{dominant['role']}",
    ])


    for contributor in (
        dominant[
            "top2_contributors"
        ]
    ):

        lines.append(

            f"Rank {contributor['rank']} | "
            f"Feature={contributor['feature']} | "
            f"Value={contributor['value']} | "
            f"SHAP={contributor['signed_shap_text']} | "
            f"|SHAP|={contributor['absolute_shap_text']} | "
            f"Direction={contributor['direction']}"
        )


    lines.extend([

        (
            "Top-2 absolute SHAP sum "
            "(copy exactly): "
            f"{dominant['top2_sum_text']}"
        ),

        (
            "Dominant-role interpretation "
            "(copy exactly): "
            +
            dominant[
                "interpretation"
            ]
        ),

        "",

        "OVERALL EXPLANATION RULE:",

        (
            "Write exactly two sentences."
        ),

        (
            "Sentence 1: state only that supplied positive "
            "SHAP evidence was toward the predicted class "
            "and supplied negative SHAP evidence, if present, "
            "was away from it."
        ),

        (
            "Sentence 2: copy the dominant-role interpretation "
            "sentence exactly."
        ),

        (
            "Do not repeat Top-2 feature names in the Overall "
            "explanation."
        ),

        "",

        (
            "Generate only the required structure from the "
            "SYSTEM prompt."
        ),

        (
            "Do not paraphrase decoded values or the "
            "dominant-role interpretation."
        ),
    ])


    return "\n".join(
        lines
    )


# ============================================================
# FUNCTION VALIDATION
# ============================================================

assert callable(
    compact_feature
)

assert callable(
    compact_dominant_contributor
)

assert callable(
    compact_dominant_role
)

assert callable(
    build_user_prompt
)

assert isinstance(
    SYSTEM_PROMPT,
    str
)

assert len(
    SYSTEM_PROMPT.strip()
) > 0

assert len(
    evidence_data
) > 0


# ============================================================
# RESPONDENT 0 TEST
# ============================================================

test_item = (
    evidence_data[
        0
    ]
)

test_dominant_role = (
    test_item[
        "dominant_role"
    ]
)


assert (
    "role"
    in test_dominant_role
)

assert (
    "top2_contributors"
    in test_dominant_role
)

assert len(
    test_dominant_role[
        "top2_contributors"
    ]
) == 2

assert (
    "top2_absolute_shap_sum"
    in test_dominant_role
)

assert (
    "interpretation"
    in test_dominant_role
)


test_prompt = (
    build_user_prompt(
        test_item
    )
)


# ============================================================
# ANTI-HINDSIGHT LEAKAGE CHECK
# ============================================================

assert (
    "true_class"
    not in test_prompt
)

assert (
    '"correct"'
    not in test_prompt
)

assert (
    "role_summary"
    not in test_prompt
)


# ============================================================
# GENERAL EVIDENCE CHECK
# ============================================================

for record in (

    test_item.get(
        "supporting_evidence",
        []
    )

    +

    test_item.get(
        "opposing_evidence",
        []
    )

):

    assert (
        str(
            record[
                "feature"
            ]
        )
        in test_prompt
    )

    assert (
        str(
            record[
                "decoded_value"
            ]
        )
        in test_prompt
    )

    assert (

        f"{float(record['shap_value']):+.6f}"

        in test_prompt
    )


# ============================================================
# DOMINANT-ROLE CHECK
# ============================================================

assert (
    str(
        test_dominant_role[
            "role"
        ]
    )
    in test_prompt
)


for contributor in (

    test_dominant_role[
        "top2_contributors"
    ]

):

    assert (
        str(
            contributor[
                "feature"
            ]
        )
        in test_prompt
    )

    assert (
        str(
            contributor[
                "decoded_value"
            ]
        )
        in test_prompt
    )

    assert (

        f"{float(contributor['signed_shap']):+.6f}"

        in test_prompt
    )

    assert (

        f"{float(contributor['absolute_shap']):.6f}"

        in test_prompt
    )


# ============================================================
# TOP-2 SUM CHECK
# ============================================================

expected_top2_sum_text = (

    f"{float(
        test_dominant_role[
            'top2_absolute_shap_sum'
        ]
    ):.6f}"
)


assert (
    expected_top2_sum_text
    in test_prompt
)


# ============================================================
# INTERPRETATION CHECK
# ============================================================

assert (
    test_dominant_role[
        "interpretation"
    ]
    in test_prompt
)


# ============================================================
# MEMORY / TOKEN DIAGNOSTIC
# ============================================================

messages_preview = [

    {
        "role":
            "system",

        "content":
            SYSTEM_PROMPT
    },

    {
        "role":
            "user",

        "content":
            test_prompt
    },
]


try:

    chat_preview = (
        tokenizer.apply_chat_template(

            messages_preview,

            tokenize=False,

            add_generation_prompt=True,

            enable_thinking=False,
        )
    )

except TypeError:

    chat_preview = (
        tokenizer.apply_chat_template(

            messages_preview,

            tokenize=False,

            add_generation_prompt=True,
        )
    )


prompt_token_count = len(

    tokenizer(

        chat_preview,

        add_special_tokens=False,

    )[
        "input_ids"
    ]
)


# ============================================================
# PRINT VALIDATION
# ============================================================

print(
    "=" * 90
)

print(
    "FINAL MEMORY-OPTIMIZED SHAP-GROUNDED QWEN PROMPT"
)

print(
    "=" * 90
)


print(
    "SYSTEM_PROMPT chars :",
    len(
        SYSTEM_PROMPT
    )
)


print(
    "Respondent 0 chars  :",
    len(
        test_prompt
    )
)


print(
    "Total prompt tokens :",
    prompt_token_count
)


print(
    "Dominant role       :",
    test_dominant_role[
        "role"
    ]
)


print(
    "Top-2 sum           :",
    expected_top2_sum_text
)


print(
    "\nTop-2 contributors:"
)


for contributor in (

    test_dominant_role[
        "top2_contributors"
    ]

):

    print(

        f" - "
        f"{contributor['feature']} | "
        f"Value="
        f"{contributor['decoded_value']} | "
        f"SHAP="
        f"{float(contributor['signed_shap']):+.6f} | "
        f"|SHAP|="
        f"{float(contributor['absolute_shap']):.6f}"
    )


if prompt_token_count > 4000:

    print(

        "\nWARNING: prompt exceeds 4000 tokens. "
        "Do not run the batch until prompt size is reviewed."
    )

else:

    print(
        "\nPrompt-size check: PASSED"
    )


print(
    "\nFINAL PROMPT BUILDER VALIDATION: PASSED"
)


FINAL MEMORY-OPTIMIZED SHAP-GROUNDED QWEN PROMPT
SYSTEM_PROMPT chars : 5248
Respondent 0 chars  : 1955
Total prompt tokens : 1733
Dominant role       : Economic
Top-2 sum           : 0.144746

Top-2 contributors:
 - Average Expendature for transportation | Value=1000 BDT/month | SHAP=+0.083161 | |SHAP|=0.083161
 - Monthly Income | Value=0 BDT/month | SHAP=+0.061585 | |SHAP|=0.061585

Prompt-size check: PASSED

FINAL PROMPT BUILDER VALIDATION: PASSED


## Deterministic generation + strict fidelity audit

In [ ]:
# ============================================================
# CELL F — JOURNAL-READY QWEN GENERATION + CORE FIDELITY AUDIT
# NON-BLOCKING SEMANTIC FLAGS
# LOCAL TOP-2 DOMINANT-ROLE EXPLANATION
# ============================================================

import gc
import re
import time

import numpy as np
import torch


# ============================================================
# GENERATION SETTINGS
# ============================================================

MAX_NEW_TOKENS = 700


# ============================================================
# REQUIRED OUTPUT STRUCTURE
# ============================================================

REQUIRED_SECTIONS = [
    "Predicted mode:",
    "Prediction probability:",
    "Runner-up mode:",
    "Runner-up probability:",
    "Probability margin:",
    "Supporting evidence:",
    "Opposing evidence:",
    "Uncertainty note:",
    "Dominant role:",
    "Top-2 contributors within dominant role:",
    "Dominant-role Top-2 absolute SHAP sum:",
    "Dominant-role interpretation:",
    "Overall explanation:",
]


# ============================================================
# SEMANTIC WARNING LISTS
#
# IMPORTANT:
# These are REVIEW FLAGS.
# They do NOT automatically invalidate a factually faithful
# explanation.
# ============================================================

PROHIBITED_CAUSAL_PHRASES = [
    "caused the prediction",
    "caused the transport choice",
    "caused the choice",
    "determined the prediction",
    "determined the choice",
    "determined the transport choice",
    "resulted in the mode",
    "resulted in the choice",
    "led the respondent to choose",
    "made the respondent choose",
    "responsible for the prediction",
    "responsible for the choice",
    "responsible for the transport choice",
    "drove the respondent's behaviour",
    "drove the respondent’s behaviour",
    "drove the respondent's behavior",
    "drove the respondent’s behavior",
    "causal driver",
]


PROHIBITED_ROLE_OVERCLAIMS = [
    "strongest influence",
    "strongest local influence",
    "strongest overall influence",
    "strongest local evidence",
    "strongest evidence",
    "strongest model evidence",
    "most influential role",
    "most influential factor",
    "most significant role",
    "most significant factor",
    "most important role",
    "most important factor",
    "main driver",
    "primary driver",
    "dominant behavioural factor",
    "dominant behavioral factor",
    "greatest behavioural effect",
    "greatest behavioral effect",
    "played the most significant part",
    "played the most important part",
    "had the strongest influence",
    "had the strongest local influence",
    "had the strongest evidence",
    "had the greatest influence",
    "had the most influence",
]


PROHIBITED_BEHAVIOURAL_PHRASES = [
    "indicating a preference",
    "indicates a preference",
    "reflecting a preference",
    "reflects a preference",
    "showing a preference",
    "shows a preference",
    "preference for intermediate",
    "preference for public",
    "preference for low-cost personal",
    "preference for private/hired",
    "preferred mode",
    "respondent preferred",
    "respondent prefers",
    "respondent's preference",
    "respondent’s preference",
    "behavioural preference",
    "behavioral preference",
    "tendency to choose",
    "tendency toward",
    "habitual choice",
    "emotional preference",
]


PROHIBITED_MAGNITUDE_PHRASES = [
    "minor contribution",
    "minor opposing contribution",
    "minor positive contribution",
    "major contribution",
    "slight contribution",
    "slightly influenced",
    "slightly shifted",
    "strong contribution",
    "weak contribution",
    "substantial contribution",
    "negligible contribution",
    "significant contribution",
    "notable contribution",
    "smaller negative contribution",
    "smaller negative contributions",
    "smaller contribution",
    "larger contribution",
]


PROHIBITED_VALUE_PARAPHRASES = [
    "high monthly income",
    "low monthly income",
    "moderate monthly income",
    "high income",
    "low income",
    "moderate income",
    "high transportation expenditure",
    "low transportation expenditure",
    "moderate transportation expenditure",
    "high expenditure",
    "low expenditure",
    "moderate expenditure",
]


PROHIBITED_UNCERTAINTY_PHRASES = [
    "no uncertainty information was provided",
    "uncertainty information is unavailable",
    "no additional uncertainty information provided",
]


PROHIBITED_UNCERTAINTY_QUALIFIERS = [
    "notable separation",
    "substantial separation",
    "strong separation",
    "weak separation",
    "large separation",
    "small separation",
    "considerable separation",
    "large margin",
    "small margin",
    "high confidence",
    "medium confidence",
    "low confidence",
    "strong confidence",
    "weak confidence",
]


# ============================================================
# QWEN GENERATION
# ============================================================

def generate_qwen_explanation(item):

    # --------------------------------------------------------
    # MEMORY HYGIENE
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    user_prompt = build_user_prompt(item)

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    try:

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

    except TypeError:

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    model_inputs = tokenizer(
        text,
        return_tensors="pt",
    )

    model_inputs = {
        key: value.to("cuda:0")
        for key, value in model_inputs.items()
    }

    input_length = (
        model_inputs["input_ids"].shape[1]
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()

    with torch.inference_mode():

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed_time = (
        time.time() - start_time
    )

    new_tokens = generated_ids[
        0,
        input_length:
    ]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    n_generated_tokens = int(
        new_tokens.shape[0]
    )

    result = {

        "response":
            response,

        "generated_tokens":
            n_generated_tokens,

        "generation_seconds":
            float(elapsed_time),

        "tokens_per_second":
            float(
                n_generated_tokens / elapsed_time
                if elapsed_time > 0
                else 0
            ),

        "hit_token_limit":
            bool(
                n_generated_tokens
                >= MAX_NEW_TOKENS
            ),
    }


    # --------------------------------------------------------
    # REMOVE TEMPORARY GPU TENSORS
    # --------------------------------------------------------

    del model_inputs
    del generated_ids
    del new_tokens

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# NUMERIC / TEXT HELPERS
# ============================================================

def _number_forms(
    value,
    precision=4,
):

    value = float(value)

    forms = {
        str(value),
        f"{value:.{precision}f}",
        f"{value:g}",
    }

    return {
        form
        for form in forms
        if form
    }


def _signed_shap_text(value):

    return (
        f"{float(value):+.6f}"
    )


def _absolute_shap_text(value):

    return (
        f"{abs(float(value)):.6f}"
    )


def _six_decimal_text(value):

    return (
        f"{float(value):.6f}"
    )


def _normalize_space(text):

    return re.sub(
        r"\s+",
        " ",
        str(text).strip(),
    )


def _contains_exact_term(
    text,
    term,
):

    return (
        str(term).lower()
        in
        str(text).lower()
    )


# ============================================================
# SECTION EXTRACTION
# ============================================================

def _extract_section(
    response,
    heading,
    next_headings,
):

    escaped_heading = (
        re.escape(heading)
    )

    lookahead_parts = [
        re.escape(next_heading)
        for next_heading in next_headings
    ]

    if lookahead_parts:

        lookahead = (
            r"(?=\n\s*(?:"
            +
            "|".join(
                lookahead_parts
            )
            +
            r")|\Z)"
        )

    else:

        lookahead = r"(?=\Z)"

    pattern = (
        escaped_heading
        +
        r"\s*(.*?)"
        +
        lookahead
    )

    match = re.search(
        pattern,
        response,
        flags=(
            re.IGNORECASE
            |
            re.DOTALL
        ),
    )

    if match:

        return (
            match
            .group(1)
            .strip()
        )

    return ""


# ============================================================
# SENTENCE HELPER
# ============================================================

def _split_sentences(text):

    text = str(text).strip()

    if not text:
        return []

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text,
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


# ============================================================
# NUMERIC-EQUIVALENCE HELPER
# ============================================================

def _numeric_value_present(
    text,
    expected_value,
    atol=5e-7,
):

    numbers = re.findall(
        (
            r"[-+]?"
            r"(?:\d+(?:\.\d*)?|\.\d+)"
            r"(?:[eE][-+]?\d+)?"
        ),
        str(text),
    )

    for number in numbers:

        try:

            if np.isclose(
                float(number),
                float(expected_value),
                atol=atol,
                rtol=0,
            ):

                return True

        except Exception:
            pass

    return False


# ============================================================
# JOURNAL-READY CORE FIDELITY AUDIT
#
# audit_pass:
#   factual / structural fidelity only
#
# audit_flagged:
#   semantic or wording issue worth reviewing
#
# Semantic warnings DO NOT automatically set audit_pass=False.
# ============================================================

def audit_qwen_explanation(
    item,
    generation,
):

    response = (
        generation["response"]
    )

    response_lower = (
        response.lower()
    )

    model_output = (
        item["model_output"]
    )


    # ========================================================
    # 1. REQUIRED STRUCTURE
    # ========================================================

    missing_sections = [

        section

        for section
        in REQUIRED_SECTIONS

        if section.lower()
        not in response_lower
    ]


    # ========================================================
    # 2. EXTRACT SECTIONS
    # ========================================================

    predicted_mode_section = (
        _extract_section(
            response,
            "Predicted mode:",
            [
                "Prediction probability:",
                "Runner-up mode:",
                "Runner-up probability:",
                "Probability margin:",
                "Supporting evidence:",
            ],
        )
    )


    prediction_probability_section = (
        _extract_section(
            response,
            "Prediction probability:",
            [
                "Runner-up mode:",
                "Runner-up probability:",
                "Probability margin:",
                "Supporting evidence:",
            ],
        )
    )


    runner_up_mode_section = (
        _extract_section(
            response,
            "Runner-up mode:",
            [
                "Runner-up probability:",
                "Probability margin:",
                "Supporting evidence:",
            ],
        )
    )


    runner_up_probability_section = (
        _extract_section(
            response,
            "Runner-up probability:",
            [
                "Probability margin:",
                "Supporting evidence:",
            ],
        )
    )


    probability_margin_section = (
        _extract_section(
            response,
            "Probability margin:",
            [
                "Supporting evidence:",
            ],
        )
    )


    supporting_section = (
        _extract_section(
            response,
            "Supporting evidence:",
            [
                "Opposing evidence:",
                "Uncertainty note:",
                "Dominant role:",
                "Top-2 contributors within dominant role:",
                "Dominant-role Top-2 absolute SHAP sum:",
                "Dominant-role interpretation:",
                "Overall explanation:",
            ],
        )
    )


    opposing_section = (
        _extract_section(
            response,
            "Opposing evidence:",
            [
                "Uncertainty note:",
                "Dominant role:",
                "Top-2 contributors within dominant role:",
                "Dominant-role Top-2 absolute SHAP sum:",
                "Dominant-role interpretation:",
                "Overall explanation:",
            ],
        )
    )


    uncertainty_section = (
        _extract_section(
            response,
            "Uncertainty note:",
            [
                "Dominant role:",
                "Top-2 contributors within dominant role:",
                "Dominant-role Top-2 absolute SHAP sum:",
                "Dominant-role interpretation:",
                "Overall explanation:",
            ],
        )
    )


    dominant_role_section = (
        _extract_section(
            response,
            "Dominant role:",
            [
                "Top-2 contributors within dominant role:",
                "Dominant-role Top-2 absolute SHAP sum:",
                "Dominant-role interpretation:",
                "Overall explanation:",
            ],
        )
    )


    dominant_contributors_section = (
        _extract_section(
            response,
            "Top-2 contributors within dominant role:",
            [
                "Dominant-role Top-2 absolute SHAP sum:",
                "Dominant-role interpretation:",
                "Overall explanation:",
            ],
        )
    )


    dominant_sum_section = (
        _extract_section(
            response,
            "Dominant-role Top-2 absolute SHAP sum:",
            [
                "Dominant-role interpretation:",
                "Overall explanation:",
            ],
        )
    )


    dominant_interpretation_section = (
        _extract_section(
            response,
            "Dominant-role interpretation:",
            [
                "Overall explanation:",
            ],
        )
    )


    overall_section = (
        _extract_section(
            response,
            "Overall explanation:",
            [],
        )
    )


    # ========================================================
    # 3. FIXED MODEL OUTPUT FIDELITY
    # ========================================================

    predicted_mode = str(
        model_output[
            "predicted_mode"
        ]
    ).strip()

    runner_up_mode = str(
        model_output[
            "runner_up_mode"
        ]
    ).strip()


    predicted_mode_present = (

        _normalize_space(
            predicted_mode_section
        ).lower()

        ==

        _normalize_space(
            predicted_mode
        ).lower()
    )


    runner_up_mode_present = (

        _normalize_space(
            runner_up_mode_section
        ).lower()

        ==

        _normalize_space(
            runner_up_mode
        ).lower()
    )


    prediction_probability_present = (
        _numeric_value_present(
            prediction_probability_section,
            model_output[
                "predicted_probability"
            ],
            atol=5e-7,
        )
    )


    runner_up_probability_present = (
        _numeric_value_present(
            runner_up_probability_section,
            model_output[
                "runner_up_probability"
            ],
            atol=5e-7,
        )
    )


    probability_margin_present = (
        _numeric_value_present(
            probability_margin_section,
            model_output[
                "probability_margin"
            ],
            atol=5e-7,
        )
    )


    probability_checks = {

        "predicted_probability":
            bool(
                prediction_probability_present
            ),

        "runner_up_probability":
            bool(
                runner_up_probability_present
            ),

        "probability_margin":
            bool(
                probability_margin_present
            ),
    }


    # ========================================================
    # 4. UNCERTAINTY NUMERIC FIDELITY
    # ========================================================

    uncertainty_predicted_probability_present = (
        _numeric_value_present(
            uncertainty_section,
            model_output[
                "predicted_probability"
            ],
            atol=5e-7,
        )
    )


    uncertainty_runner_up_probability_present = (
        _numeric_value_present(
            uncertainty_section,
            model_output[
                "runner_up_probability"
            ],
            atol=5e-7,
        )
    )


    uncertainty_margin_present = (
        _numeric_value_present(
            uncertainty_section,
            model_output[
                "probability_margin"
            ],
            atol=5e-7,
        )
    )


    uncertainty_numeric_fidelity_ok = all([
        uncertainty_predicted_probability_present,
        uncertainty_runner_up_probability_present,
        uncertainty_margin_present,
    ])


    # ========================================================
    # 5. GENERAL SHAP EVIDENCE FIDELITY
    # ========================================================

    supporting_records = item.get(
        "supporting_evidence",
        [],
    )

    opposing_records = item.get(
        "opposing_evidence",
        [],
    )


    missing_feature_names = []
    missing_signed_shap = []
    missing_decoded_values = []
    missing_directions = []


    # --------------------------------------------------------
    # Supporting
    # --------------------------------------------------------

    for record in supporting_records:

        feature = str(
            record[
                "feature"
            ]
        )

        decoded_value = str(
            record[
                "decoded_value"
            ]
        )

        signed_shap = (
            _signed_shap_text(
                record[
                    "shap_value"
                ]
            )
        )

        direction = str(
            record[
                "direction"
            ]
        )


        if not _contains_exact_term(
            supporting_section,
            feature,
        ):

            missing_feature_names.append({
                "section":
                    "supporting",
                "feature":
                    feature,
            })


        if not _contains_exact_term(
            supporting_section,
            decoded_value,
        ):

            missing_decoded_values.append({
                "section":
                    "supporting",
                "feature":
                    feature,
                "expected_value":
                    decoded_value,
            })


        if (
            signed_shap
            not in supporting_section
        ):

            missing_signed_shap.append({
                "section":
                    "supporting",
                "feature":
                    feature,
                "expected_shap":
                    signed_shap,
            })


        if not _contains_exact_term(
            supporting_section,
            direction,
        ):

            missing_directions.append({
                "section":
                    "supporting",
                "feature":
                    feature,
                "expected_direction":
                    direction,
            })


    # --------------------------------------------------------
    # Opposing
    # --------------------------------------------------------

    for record in opposing_records:

        feature = str(
            record[
                "feature"
            ]
        )

        decoded_value = str(
            record[
                "decoded_value"
            ]
        )

        signed_shap = (
            _signed_shap_text(
                record[
                    "shap_value"
                ]
            )
        )

        direction = str(
            record[
                "direction"
            ]
        )


        if not _contains_exact_term(
            opposing_section,
            feature,
        ):

            missing_feature_names.append({
                "section":
                    "opposing",
                "feature":
                    feature,
            })


        if not _contains_exact_term(
            opposing_section,
            decoded_value,
        ):

            missing_decoded_values.append({
                "section":
                    "opposing",
                "feature":
                    feature,
                "expected_value":
                    decoded_value,
            })


        if (
            signed_shap
            not in opposing_section
        ):

            missing_signed_shap.append({
                "section":
                    "opposing",
                "feature":
                    feature,
                "expected_shap":
                    signed_shap,
            })


        if not _contains_exact_term(
            opposing_section,
            direction,
        ):

            missing_directions.append({
                "section":
                    "opposing",
                "feature":
                    feature,
                "expected_direction":
                    direction,
            })


    general_shap_fidelity_ok = all([

        len(
            missing_feature_names
        ) == 0,

        len(
            missing_decoded_values
        ) == 0,

        len(
            missing_signed_shap
        ) == 0,

        len(
            missing_directions
        ) == 0,
    ])


    # ========================================================
    # 6. PRE-COMPUTED DOMINANT ROLE
    # ========================================================

    dominant_role_info = (
        item.get(
            "dominant_role",
            {},
        )
        or {}
    )


    dominant_role_name = str(
        dominant_role_info.get(
            "role",
            "",
        )
    ).strip()


    dominant_top2 = (
        dominant_role_info.get(
            "top2_contributors",
            [],
        )
        or []
    )


    dominant_top2_sum = (
        dominant_role_info.get(
            "top2_absolute_shap_sum",
            None,
        )
    )


    dominant_interpretation = str(
        dominant_role_info.get(
            "interpretation",
            "",
        )
    ).strip()


    dominant_role_present = (

        _normalize_space(
            dominant_role_section
        ).lower()

        ==

        _normalize_space(
            dominant_role_name
        ).lower()

        if dominant_role_name

        else False
    )


    # ========================================================
    # 7. DOMINANT TOP-2 FIDELITY
    # ========================================================

    dominant_top2_count_ok = (
        len(
            dominant_top2
        ) == 2
    )


    missing_dominant_top2_features = []
    missing_dominant_top2_values = []
    missing_dominant_top2_signed_shap = []
    missing_dominant_top2_absolute_shap = []
    dominant_top2_direction_errors = []


    for contributor in dominant_top2:

        feature = str(
            contributor[
                "feature"
            ]
        )

        decoded_value = str(
            contributor[
                "decoded_value"
            ]
        )

        signed_shap_text = (
            _signed_shap_text(
                contributor[
                    "signed_shap"
                ]
            )
        )

        absolute_shap_text = (
            _absolute_shap_text(
                contributor[
                    "absolute_shap"
                ]
            )
        )

        direction = str(
            contributor[
                "direction"
            ]
        )


        if not _contains_exact_term(
            dominant_contributors_section,
            feature,
        ):

            missing_dominant_top2_features.append(
                feature
            )


        if not _contains_exact_term(
            dominant_contributors_section,
            decoded_value,
        ):

            missing_dominant_top2_values.append(
                decoded_value
            )


        if (
            signed_shap_text
            not in dominant_contributors_section
        ):

            missing_dominant_top2_signed_shap.append(
                signed_shap_text
            )


        if (
            absolute_shap_text
            not in dominant_contributors_section
        ):

            missing_dominant_top2_absolute_shap.append(
                absolute_shap_text
            )


        if not _contains_exact_term(
            dominant_contributors_section,
            direction,
        ):

            dominant_top2_direction_errors.append({
                "feature":
                    feature,

                "expected_direction":
                    direction,
            })


    dominant_top2_fidelity_ok = all([

        dominant_top2_count_ok,

        len(
            missing_dominant_top2_features
        ) == 0,

        len(
            missing_dominant_top2_values
        ) == 0,

        len(
            missing_dominant_top2_signed_shap
        ) == 0,

        len(
            missing_dominant_top2_absolute_shap
        ) == 0,

        len(
            dominant_top2_direction_errors
        ) == 0,
    ])


    # ========================================================
    # 8. TOP-2 ABSOLUTE SHAP SUM
    #
    # Numeric-equivalent reporting accepted:
    #
    # 0.150150 == 0.15015
    # ========================================================

    if dominant_top2_sum is not None:

        expected_top2_sum_text = (
            _six_decimal_text(
                dominant_top2_sum
            )
        )

        dominant_top2_sum_present = (
            _numeric_value_present(
                dominant_sum_section,
                dominant_top2_sum,
                atol=5e-7,
            )
        )

    else:

        expected_top2_sum_text = ""

        dominant_top2_sum_present = False


    # ========================================================
    # 9. DOMINANT-ROLE INTERPRETATION
    # ========================================================

    dominant_interpretation_present = (

        _normalize_space(
            dominant_interpretation_section
        ).lower()

        ==

        _normalize_space(
            dominant_interpretation
        ).lower()

        if dominant_interpretation

        else False
    )


    # ========================================================
    # 10. EMPTY OPPOSING-EVIDENCE RULE
    # ========================================================

    empty_opposing_rule_ok = True


    if len(
        opposing_records
    ) == 0:

        expected_sentence = (
            "No opposing feature passed the supplied "
            "evidence inclusion threshold."
        )

        empty_opposing_rule_ok = (
            expected_sentence.lower()
            in opposing_section.lower()
        )


    # ========================================================
    # 11. OVERALL EXPLANATION
    #
    # INFORMATIONAL DIAGNOSTIC ONLY.
    # NOT A HARD-FAIL CONDITION.
    # ========================================================

    overall_sentences = (
        _split_sentences(
            overall_section
        )
    )

    overall_sentence_count = (
        len(
            overall_sentences
        )
    )


    overall_sentence_count_ok = (
        overall_sentence_count == 2
    )


    overall_interpretation_present = (

        _normalize_space(
            dominant_interpretation
        ).lower()

        in

        _normalize_space(
            overall_section
        ).lower()

        if dominant_interpretation

        else False
    )


    overall_ends_with_interpretation = (

        _normalize_space(
            overall_section
        ).lower().endswith(

            _normalize_space(
                dominant_interpretation
            ).lower()
        )

        if dominant_interpretation

        else False
    )


    # Keep legacy keys for Cell H / I compatibility.

    overall_direction_summary_ok = True

    overall_second_sentence_exact = (
        overall_ends_with_interpretation
    )

    overall_interpretation_occurrences = (
        _normalize_space(
            overall_section
        ).lower().count(

            _normalize_space(
                dominant_interpretation
            ).lower()
        )

        if dominant_interpretation

        else 0
    )

    overall_interpretation_once = (
        overall_interpretation_occurrences
        == 1
    )


    overall_feature_mentions = []


    all_evidence_records = (
        supporting_records
        +
        opposing_records
    )


    for record in all_evidence_records:

        feature = str(
            record[
                "feature"
            ]
        )

        if _contains_exact_term(
            overall_section,
            feature,
        ):

            overall_feature_mentions.append(
                feature
            )


    overall_no_feature_mentions = (
        len(
            overall_feature_mentions
        ) == 0
    )


    # ========================================================
    # 12. SEMANTIC REVIEW FLAGS
    #
    # These are NON-BLOCKING.
    # ========================================================

    causal_violations = [

        phrase

        for phrase
        in PROHIBITED_CAUSAL_PHRASES

        if phrase
        in response_lower
    ]


    role_overclaim_violations = [

        phrase

        for phrase
        in PROHIBITED_ROLE_OVERCLAIMS

        if phrase
        in response_lower
    ]


    behavioural_violations = [

        phrase

        for phrase
        in PROHIBITED_BEHAVIOURAL_PHRASES

        if phrase
        in response_lower
    ]


    magnitude_violations = [

        phrase

        for phrase
        in PROHIBITED_MAGNITUDE_PHRASES

        if phrase
        in response_lower
    ]


    value_paraphrase_violations = [

        phrase

        for phrase
        in PROHIBITED_VALUE_PARAPHRASES

        if phrase
        in response_lower
    ]


    uncertainty_violations = [

        phrase

        for phrase
        in PROHIBITED_UNCERTAINTY_PHRASES

        if phrase
        in response_lower
    ]


    uncertainty_qualifier_violations = [

        phrase

        for phrase
        in PROHIBITED_UNCERTAINTY_QUALIFIERS

        if phrase
        in uncertainty_section.lower()
    ]


    semantic_warning_count = sum([

        len(
            causal_violations
        ),

        len(
            role_overclaim_violations
        ),

        len(
            behavioural_violations
        ),

        len(
            magnitude_violations
        ),

        len(
            value_paraphrase_violations
        ),

        len(
            uncertainty_violations
        ),

        len(
            uncertainty_qualifier_violations
        ),
    ])


    semantic_warning = (
        semantic_warning_count > 0
    )


    # ========================================================
    # 13. CORE JOURNAL FIDELITY FAILURES
    #
    # ONLY THESE determine audit_pass.
    # ========================================================

    core_failures = []


    if len(
        missing_sections
    ) > 0:

        core_failures.append(
            "missing_required_section"
        )


    if not predicted_mode_present:

        core_failures.append(
            "predicted_mode_mismatch"
        )


    if not runner_up_mode_present:

        core_failures.append(
            "runner_up_mode_mismatch"
        )


    if not all(
        probability_checks.values()
    ):

        core_failures.append(
            "prediction_probability_mismatch"
        )


    if not uncertainty_numeric_fidelity_ok:

        core_failures.append(
            "uncertainty_numeric_mismatch"
        )


    if not general_shap_fidelity_ok:

        core_failures.append(
            "general_shap_evidence_mismatch"
        )


    if not dominant_role_present:

        core_failures.append(
            "dominant_role_mismatch"
        )


    if not dominant_top2_count_ok:

        core_failures.append(
            "dominant_top2_count_error"
        )


    if not dominant_top2_fidelity_ok:

        core_failures.append(
            "dominant_top2_fidelity_error"
        )


    if not dominant_top2_sum_present:

        core_failures.append(
            "dominant_top2_sum_mismatch"
        )


    if not dominant_interpretation_present:

        core_failures.append(
            "dominant_interpretation_mismatch"
        )


    if not empty_opposing_rule_ok:

        core_failures.append(
            "empty_opposing_rule_error"
        )


    if generation[
        "hit_token_limit"
    ]:

        core_failures.append(
            "generation_truncated"
        )


    # ========================================================
    # 14. FINAL STATUS
    #
    # audit_pass:
    # factual fidelity
    #
    # audit_flagged:
    # anything deserving inspection
    # ========================================================

    audit_pass = (
        len(
            core_failures
        ) == 0
    )


    overall_warning = any([

        not overall_sentence_count_ok,

        not overall_interpretation_present,

        not overall_ends_with_interpretation,
    ])


    audit_flagged = any([

        not audit_pass,

        semantic_warning,

        overall_warning,
    ])


    review_recommended = (
        audit_flagged
    )


    # ========================================================
    # 15. RETURN DIAGNOSTICS
    # ========================================================

    return {

        # ----------------------------------------------------
        # Main status
        # ----------------------------------------------------

        "audit_pass":
            bool(
                audit_pass
            ),

        "journal_core_pass":
            bool(
                audit_pass
            ),

        "audit_flagged":
            bool(
                audit_flagged
            ),

        "review_recommended":
            bool(
                review_recommended
            ),

        "core_failures":
            core_failures,


        # ----------------------------------------------------
        # Structure
        # ----------------------------------------------------

        "missing_sections":
            missing_sections,


        # ----------------------------------------------------
        # RF output fidelity
        # ----------------------------------------------------

        "predicted_mode_present":
            bool(
                predicted_mode_present
            ),

        "runner_up_mode_present":
            bool(
                runner_up_mode_present
            ),

        "probability_checks":
            probability_checks,

        "uncertainty_numeric_fidelity_ok":
            bool(
                uncertainty_numeric_fidelity_ok
            ),


        # ----------------------------------------------------
        # General SHAP evidence
        # ----------------------------------------------------

        "general_shap_fidelity_ok":
            bool(
                general_shap_fidelity_ok
            ),

        "missing_feature_names":
            missing_feature_names,

        "missing_signed_shap":
            missing_signed_shap,

        "missing_decoded_values":
            missing_decoded_values,

        "missing_directions":
            missing_directions,

        "supporting_section":
            supporting_section,

        "opposing_section":
            opposing_section,


        # ----------------------------------------------------
        # Dominant role
        # ----------------------------------------------------

        "dominant_role_expected":
            dominant_role_name,

        "dominant_role_present":
            bool(
                dominant_role_present
            ),

        "dominant_role_section":
            dominant_role_section,


        # ----------------------------------------------------
        # Dominant Top-2
        # ----------------------------------------------------

        "dominant_top2_count_ok":
            bool(
                dominant_top2_count_ok
            ),

        "dominant_top2_fidelity_ok":
            bool(
                dominant_top2_fidelity_ok
            ),

        "missing_dominant_top2_features":
            missing_dominant_top2_features,

        "missing_dominant_top2_values":
            missing_dominant_top2_values,

        "missing_dominant_top2_signed_shap":
            missing_dominant_top2_signed_shap,

        "missing_dominant_top2_absolute_shap":
            missing_dominant_top2_absolute_shap,

        "dominant_top2_direction_errors":
            dominant_top2_direction_errors,

        "dominant_contributors_section":
            dominant_contributors_section,


        # ----------------------------------------------------
        # Top-2 sum
        # ----------------------------------------------------

        "dominant_top2_sum_expected":
            expected_top2_sum_text,

        "dominant_top2_sum_present":
            bool(
                dominant_top2_sum_present
            ),

        "dominant_sum_section":
            dominant_sum_section,


        # ----------------------------------------------------
        # Interpretation
        # ----------------------------------------------------

        "dominant_interpretation_expected":
            dominant_interpretation,

        "dominant_interpretation_present":
            bool(
                dominant_interpretation_present
            ),

        "dominant_interpretation_section":
            dominant_interpretation_section,


        # ----------------------------------------------------
        # Overall — informational only
        # ----------------------------------------------------

        "overall_sentence_count":
            int(
                overall_sentence_count
            ),

        "overall_sentence_count_ok":
            bool(
                overall_sentence_count_ok
            ),

        "overall_interpretation_present":
            bool(
                overall_interpretation_present
            ),

        "overall_ends_with_interpretation":
            bool(
                overall_ends_with_interpretation
            ),

        # Legacy compatibility
        "overall_direction_summary_ok":
            bool(
                overall_direction_summary_ok
            ),

        "overall_second_sentence_exact":
            bool(
                overall_second_sentence_exact
            ),

        "overall_interpretation_occurrences":
            int(
                overall_interpretation_occurrences
            ),

        "overall_interpretation_once":
            bool(
                overall_interpretation_once
            ),

        "overall_no_feature_mentions":
            bool(
                overall_no_feature_mentions
            ),

        "overall_feature_mentions":
            overall_feature_mentions,

        "overall_section":
            overall_section,


        # ----------------------------------------------------
        # Non-blocking semantic flags
        # ----------------------------------------------------

        "semantic_warning":
            bool(
                semantic_warning
            ),

        "semantic_warning_count":
            int(
                semantic_warning_count
            ),

        "causal_violations":
            causal_violations,

        "role_overclaim_violations":
            role_overclaim_violations,

        "behavioural_violations":
            behavioural_violations,

        "magnitude_violations":
            magnitude_violations,

        "value_paraphrase_violations":
            value_paraphrase_violations,

        "uncertainty_violations":
            uncertainty_violations,

        "uncertainty_qualifier_violations":
            uncertainty_qualifier_violations,


        # ----------------------------------------------------
        # Other
        # ----------------------------------------------------

        "empty_opposing_rule_ok":
            bool(
                empty_opposing_rule_ok
            ),

        "hit_token_limit":
            bool(
                generation[
                    "hit_token_limit"
                ]
            ),
    }


# ============================================================
# READY
# ============================================================

print(
    "=" * 90
)

print(
    "JOURNAL-READY QWEN GENERATION + CORE FIDELITY AUDIT: READY"
)

print(
    "=" * 90
)

print(
    "MAX_NEW_TOKENS:",
    MAX_NEW_TOKENS,
)

print(
    "Required sections:",
    len(
        REQUIRED_SECTIONS
    ),
)

print(
    "\nHARD-FAIL / CORE CHECKS:"
)

print(
    " - required output sections"
)

print(
    " - fixed RF predicted mode"
)

print(
    " - fixed runner-up mode"
)

print(
    " - prediction probabilities / margin"
)

print(
    " - uncertainty numeric fidelity"
)

print(
    " - supporting / opposing feature fidelity"
)

print(
    " - decoded feature values"
)

print(
    " - signed SHAP values"
)

print(
    " - SHAP directions"
)

print(
    " - fixed Local Top-2 dominant role"
)

print(
    " - exact Top-2 contributor evidence"
)

print(
    " - numeric-equivalent Top-2 absolute SHAP sum"
)

print(
    " - exact dominant-role interpretation"
)

print(
    " - empty-opposing rule"
)

print(
    " - token-limit truncation"
)


print(
    "\nNON-BLOCKING REVIEW FLAGS:"
)

print(
    " - Overall explanation formatting"
)

print(
    " - causal wording"
)

print(
    " - behavioural / preference wording"
)

print(
    " - dominant-role overclaim wording"
)

print(
    " - unsupplied magnitude wording"
)

print(
    " - value paraphrasing"
)

print(
    " - uncertainty qualitative wording"
)


print(
    "\nIMPORTANT:"
)

print(
    " audit_pass = factual / structural fidelity only"
)

print(
    " audit_flagged = semantic or stylistic issue worth review"
)

print(
    " semantic warnings do NOT automatically invalidate an explanation"
)


JOURNAL-READY QWEN GENERATION + CORE FIDELITY AUDIT: READY
MAX_NEW_TOKENS: 700
Required sections: 13

HARD-FAIL / CORE CHECKS:
 - required output sections
 - fixed RF predicted mode
 - fixed runner-up mode
 - prediction probabilities / margin
 - uncertainty numeric fidelity
 - supporting / opposing feature fidelity
 - decoded feature values
 - signed SHAP values
 - SHAP directions
 - fixed Local Top-2 dominant role
 - exact Top-2 contributor evidence
 - numeric-equivalent Top-2 absolute SHAP sum
 - exact dominant-role interpretation
 - empty-opposing rule
 - token-limit truncation

NON-BLOCKING REVIEW FLAGS:
 - Overall explanation formatting
 - causal wording
 - behavioural / preference wording
 - dominant-role overclaim wording
 - unsupplied magnitude wording
 - value paraphrasing
 - uncertainty qualitative wording

IMPORTANT:
 audit_pass = factual / structural fidelity only
 audit_flagged = semantic or stylistic issue worth review
 semantic warnings do NOT automatically invalidate an

## CUDA cleanup and prompt-size check

In [ ]:
# ============================================================
# PRE-GENERATION CUDA CLEANUP + PROMPT-SIZE CHECK
# ============================================================

import gc
import torch


gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    try:

        torch.cuda.ipc_collect()

    except Exception:

        pass


print(
    "Allocated GB:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    )
)


print(
    "Reserved GB:",
    round(
        torch.cuda.memory_reserved() / 1024**3,
        2
    )
)


# ============================================================
# RESPONDENT 0 PROMPT TOKEN COUNT
# ============================================================

item_0_preview = evidence_data[0]

user_prompt_preview = build_user_prompt(
    item_0_preview
)


messages_preview = [

    {
        "role":
            "system",

        "content":
            SYSTEM_PROMPT
    },

    {
        "role":
            "user",

        "content":
            user_prompt_preview
    }
]


try:

    chat_preview = tokenizer.apply_chat_template(

        messages_preview,

        tokenize=False,

        add_generation_prompt=True,

        enable_thinking=False
    )


except TypeError:

    chat_preview = tokenizer.apply_chat_template(

        messages_preview,

        tokenize=False,

        add_generation_prompt=True
    )


prompt_tokens_preview = len(

    tokenizer(

        chat_preview,

        add_special_tokens=False

    )[
        "input_ids"
    ]
)


print(
    "Respondent 0 total prompt tokens:",
    prompt_tokens_preview
)


if prompt_tokens_preview > 4000:

    print(
        "WARNING: prompt is still large for a T4-class GPU."
    )

else:

    print(
        "Prompt-size check: PASSED"
    )


print(
    "Pre-generation memory cleanup: COMPLETE"
)


Allocated GB: 5.67
Reserved GB: 5.76
Respondent 0 total prompt tokens: 1733
Prompt-size check: PASSED
Pre-generation memory cleanup: COMPLETE


## Respondent 0 quality-control test

In [ ]:
# ============================================================
# CELL G — RESPONDENT 0 EXPLANATION + STRICT AUDIT
# ============================================================

item_0 = evidence_data[0]


generation_0 = generate_qwen_explanation(
    item_0
)


audit_0 = audit_qwen_explanation(
    item_0,
    generation_0
)


# ============================================================
# EXPLANATION
# ============================================================

print("=" * 90)
print("QWEN3 — RESPONDENT 0")
print("=" * 90)

print(
    generation_0[
        "response"
    ]
)


# ============================================================
# GENERATION DIAGNOSTICS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "GENERATION DIAGNOSTICS"
)

print(
    "=" * 90
)


print(
    "Generated tokens :",
    generation_0[
        "generated_tokens"
    ]
)


print(
    "Generation time  :",
    round(
        generation_0[
            "generation_seconds"
        ],
        2
    ),
    "seconds"
)


print(
    "Tokens/second    :",
    round(
        generation_0[
            "tokens_per_second"
        ],
        2
    )
)


print(
    "Hit token limit  :",
    generation_0[
        "hit_token_limit"
    ]
)


# ============================================================
# DOMINANT-ROLE EXPECTED EVIDENCE
# ============================================================

dominant_0 = item_0[
    "dominant_role"
]


print(
    "\n" + "=" * 90
)

print(
    "EXPECTED DOMINANT-ROLE EVIDENCE"
)

print(
    "=" * 90
)


print(
    "Dominant role    :",
    dominant_0[
        "role"
    ]
)


print(
    "Top-2 contributors:"
)


for contributor in (
    dominant_0[
        "top2_contributors"
    ]
):

    print(

        f"  Rank {contributor['rank']}: "
        f"{contributor['feature']} | "
        f"Value: {contributor['decoded_value']} | "
        f"SHAP: {float(contributor['signed_shap']):+.6f} | "
        f"|SHAP|: {float(contributor['absolute_shap']):.6f} | "
        f"Direction: {contributor['direction']}"
    )


print(
    "Top-2 absolute sum:",
    f"{float(dominant_0['top2_absolute_shap_sum']):.6f}"
)


print(
    "Interpretation   :",
    dominant_0[
        "interpretation"
    ]
)


# ============================================================
# FULL EXPLANATION AUDIT
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "EXPLANATION AUDIT"
)

print(
    "=" * 90
)


print(
    json.dumps(
        audit_0,
        indent=2,
        ensure_ascii=False
    )
)


# ============================================================
# COMPACT AUDIT SUMMARY
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "AUDIT SUMMARY"
)

print(
    "=" * 90
)


print(
    "Structure complete        :",
    len(
        audit_0[
            "missing_sections"
        ]
    ) == 0
)


print(
    "Predicted mode present    :",
    audit_0[
        "predicted_mode_present"
    ]
)


print(
    "Runner-up mode present    :",
    audit_0[
        "runner_up_mode_present"
    ]
)


print(
    "Probability checks        :",
    all(
        audit_0[
            "probability_checks"
        ].values()
    )
)


print(
    "General features complete :",
    len(
        audit_0[
            "missing_feature_names"
        ]
    ) == 0
)


print(
    "General SHAP complete     :",
    len(
        audit_0[
            "missing_signed_shap"
        ]
    ) == 0
)


print(
    "Dominant role present     :",
    audit_0[
        "dominant_role_present"
    ]
)


print(
    "Top-2 count correct       :",
    audit_0[
        "dominant_top2_count_ok"
    ]
)


print(
    "Top-2 fidelity            :",
    audit_0[
        "dominant_top2_fidelity_ok"
    ]
)


print(
    "Top-2 sum present         :",
    audit_0[
        "dominant_top2_sum_present"
    ]
)


print(
    "Interpretation present    :",
    audit_0[
        "dominant_interpretation_present"
    ]
)


print(
    "Causal violations         :",
    audit_0[
        "causal_violations"
    ]
)


print(
    "Role overclaims           :",
    audit_0[
        "role_overclaim_violations"
    ]
)


print(
    "Behavioural violations    :",
    audit_0[
        "behavioural_violations"
    ]
)


print(
    "Magnitude violations      :",
    audit_0[
        "magnitude_violations"
    ]
)


print(
    "Uncertainty violations    :",
    audit_0[
        "uncertainty_violations"
    ]
)


print(
    "Hit token limit           :",
    audit_0[
        "hit_token_limit"
    ]
)


# ============================================================
# FINAL STATUS
# ============================================================

if audit_0[
    "audit_pass"
]:

    print(
        "\nRESPONDENT 0 EXPLANATION AUDIT: PASSED"
    )

else:

    print(
        "\nRESPONDENT 0 EXPLANATION AUDIT: FAILED"
    )

    print(
        "Inspect the failed fields above before "
        "running the first-10 or full batch."
    )


QWEN3 — RESPONDENT 0
Predicted mode:
Intermediate

Prediction probability:
0.76

Runner-up mode:
Public

Runner-up probability:
0.17

Probability margin:
0.59

Supporting evidence:
- Feature=Average Expendature for transportation | Value=1000 BDT/month | SHAP=+0.083161 | Direction=toward predicted class
- Feature=Monthly Income | Value=0 BDT/month | SHAP=+0.061585 | Direction=toward predicted class
- Feature=Age | Value=22 years | SHAP=+0.043858 | Direction=toward predicted class
- Feature=Effect of Cost on Mode Choice | Value=Yes | SHAP=+0.042241 | Direction=toward predicted class
- Feature=Mode_When_Depressed | Value=Intermediate (Easybike/Rickshaw) | SHAP=+0.038284 | Direction=toward predicted class

Opposing evidence:
- Feature=Trust level based on safety in Bus | Value=Extreme | SHAP=-0.015913 | Direction=away from predicted class

Uncertainty note:
Prediction probability: 0.76 | Runner-up probability: 0.17 | Probability margin: 0.59

Dominant role:
Economic

Top-2 contributors wi

## First-10 pilot batch

In [ ]:
# ============================================================
# CELL H — FIRST 10 RESPONDENTS
# STRICT LOCAL TOP-2 DOMINANT-ROLE PILOT
# ============================================================

import os
import json
import pandas as pd


# ============================================================
# SETTINGS
# ============================================================

N_TEST = 10


FIRST10_JSON = os.path.join(
    OUTPUT_DIR,
    "qwen_first10_explanations_audited.json"
)


FIRST10_CSV = os.path.join(
    OUTPUT_DIR,
    "qwen_first10_explanations_audited.csv"
)


first10_results = []


print("=" * 90)

print(
    "QWEN3 — FIRST 10 RESPONDENTS"
)

print(
    "STRICT LOCAL TOP-2 DOMINANT-ROLE AUDIT"
)

print("=" * 90)


# ============================================================
# GENERATE FIRST 10
# ============================================================

for i in range(

    min(
        N_TEST,
        len(
            evidence_data
        )
    )

):

    item = evidence_data[
        i
    ]


    respondent_id = int(

        item[
            "respondent_id"
        ]
    )


    model_output = item[
        "model_output"
    ]


    dominant_role = item[
        "dominant_role"
    ]


    dominant_top2 = dominant_role[
        "top2_contributors"
    ]


    assert len(
        dominant_top2
    ) == 2


    print(
        f"\n[{i + 1}/{N_TEST}] "
        f"Respondent {respondent_id}"
    )


    print(
        "Expected dominant role:",
        dominant_role[
            "role"
        ]
    )


    print(
        "Expected Top-2:"
    )


    for contributor in dominant_top2:

        print(

            f"  Rank {contributor['rank']} | "
            f"{contributor['feature']} | "
            f"SHAP {float(contributor['signed_shap']):+.6f} | "
            f"|SHAP| {float(contributor['absolute_shap']):.6f}"
        )


    print(
        "Expected Top-2 sum:",
        f"{float(dominant_role['top2_absolute_shap_sum']):.6f}"
    )


    # ========================================================
    # GENERATION
    # ========================================================

    generation = generate_qwen_explanation(
        item
    )


    # ========================================================
    # STRICT AUDIT
    # ========================================================

    audit = audit_qwen_explanation(
        item,
        generation
    )


    # ========================================================
    # RESULT OBJECT
    # ========================================================

    result = {

        "respondent_id":
            respondent_id,


        # ----------------------------------------------------
        # MODEL OUTPUT
        # ----------------------------------------------------

        "predicted_mode":
            model_output[
                "predicted_mode"
            ],

        "predicted_probability":
            model_output[
                "predicted_probability"
            ],

        "runner_up_mode":
            model_output[
                "runner_up_mode"
            ],

        "runner_up_probability":
            model_output[
                "runner_up_probability"
            ],

        "probability_margin":
            model_output[
                "probability_margin"
            ],


        # ----------------------------------------------------
        # GENERAL LOCAL SHAP EVIDENCE
        # ----------------------------------------------------

        "n_supporting_features":
            len(
                item.get(
                    "supporting_evidence",
                    []
                )
            ),

        "n_opposing_features":
            len(
                item.get(
                    "opposing_evidence",
                    []
                )
            ),


        # ----------------------------------------------------
        # FINAL LOCAL TOP-2 DOMINANT ROLE
        # ----------------------------------------------------

        "dominant_role":
            dominant_role[
                "role"
            ],

        "dominant_top1_feature":
            dominant_top2[
                0
            ][
                "feature"
            ],

        "dominant_top1_value":
            dominant_top2[
                0
            ][
                "decoded_value"
            ],

        "dominant_top1_signed_shap":
            dominant_top2[
                0
            ][
                "signed_shap"
            ],

        "dominant_top1_absolute_shap":
            dominant_top2[
                0
            ][
                "absolute_shap"
            ],

        "dominant_top1_direction":
            dominant_top2[
                0
            ][
                "direction"
            ],

        "dominant_top2_feature":
            dominant_top2[
                1
            ][
                "feature"
            ],

        "dominant_top2_value":
            dominant_top2[
                1
            ][
                "decoded_value"
            ],

        "dominant_top2_signed_shap":
            dominant_top2[
                1
            ][
                "signed_shap"
            ],

        "dominant_top2_absolute_shap":
            dominant_top2[
                1
            ][
                "absolute_shap"
            ],

        "dominant_top2_direction":
            dominant_top2[
                1
            ][
                "direction"
            ],

        "dominant_role_top2_absolute_shap_sum":
            dominant_role[
                "top2_absolute_shap_sum"
            ],

        "dominant_role_interpretation":
            dominant_role[
                "interpretation"
            ],


        # ----------------------------------------------------
        # GENERATION DIAGNOSTICS
        # ----------------------------------------------------

        "generated_tokens":
            generation[
                "generated_tokens"
            ],

        "generation_seconds":
            generation[
                "generation_seconds"
            ],

        "tokens_per_second":
            generation[
                "tokens_per_second"
            ],

        "hit_token_limit":
            generation[
                "hit_token_limit"
            ],


        # ----------------------------------------------------
        # STRICT AUDIT SUMMARY
        # ----------------------------------------------------

        "audit_pass":
            audit[
                "audit_pass"
            ],

        "structure_complete":
            (
                len(
                    audit[
                        "missing_sections"
                    ]
                )
                == 0
            ),

        "predicted_mode_present":
            audit[
                "predicted_mode_present"
            ],

        "runner_up_mode_present":
            audit[
                "runner_up_mode_present"
            ],

        "probabilities_complete":
            all(
                audit[
                    "probability_checks"
                ].values()
            ),

        "general_features_complete":
            (
                len(
                    audit[
                        "missing_feature_names"
                    ]
                )
                == 0
            ),

        "general_signed_shap_complete":
            (
                len(
                    audit[
                        "missing_signed_shap"
                    ]
                )
                == 0
            ),

        "dominant_role_present":
            audit[
                "dominant_role_present"
            ],

        "dominant_top2_count_ok":
            audit[
                "dominant_top2_count_ok"
            ],

        "dominant_top2_fidelity_ok":
            audit[
                "dominant_top2_fidelity_ok"
            ],

        "dominant_top2_sum_present":
            audit[
                "dominant_top2_sum_present"
            ],

        "dominant_interpretation_present":
            audit[
                "dominant_interpretation_present"
            ],

        "causal_violation_count":
            len(
                audit[
                    "causal_violations"
                ]
            ),

        "role_overclaim_violation_count":
            len(
                audit[
                    "role_overclaim_violations"
                ]
            ),

        "behavioural_violation_count":
            len(
                audit[
                    "behavioural_violations"
                ]
            ),

        "magnitude_violation_count":
            len(
                audit[
                    "magnitude_violations"
                ]
            ),

        "uncertainty_violation_count":
            len(
                audit[
                    "uncertainty_violations"
                ]
            ),

        "empty_opposing_rule_ok":
            audit[
                "empty_opposing_rule_ok"
            ],


        # ----------------------------------------------------
        # FULL DETAILS
        # ----------------------------------------------------

        "audit_details":
            audit,

        "qwen_explanation":
            generation[
                "response"
            ]
    }


    first10_results.append(
        result
    )


    # ========================================================
    # PRINT EXPLANATION
    # ========================================================

    print(
        "\n"
        +
        generation[
            "response"
        ]
    )


    # ========================================================
    # COMPACT RESPONDENT AUDIT
    # ========================================================

    print(
        "\nStrict audit:"
    )


    print(
        "  Structure            :",
        result[
            "structure_complete"
        ]
    )


    print(
        "  General SHAP evidence:",
        (
            result[
                "general_features_complete"
            ]
            and
            result[
                "general_signed_shap_complete"
            ]
        )
    )


    print(
        "  Dominant role        :",
        result[
            "dominant_role_present"
        ]
    )


    print(
        "  Top-2 fidelity       :",
        result[
            "dominant_top2_fidelity_ok"
        ]
    )


    print(
        "  Top-2 sum            :",
        result[
            "dominant_top2_sum_present"
        ]
    )


    print(
        "  Interpretation       :",
        result[
            "dominant_interpretation_present"
        ]
    )


    print(
        "  Role overclaims      :",
        audit[
            "role_overclaim_violations"
        ]
    )


    print(
        "  Behaviour violations :",
        audit[
            "behavioural_violations"
        ]
    )


    print(
        "  Causal violations    :",
        audit[
            "causal_violations"
        ]
    )


    print(
        "  Magnitude violations :",
        audit[
            "magnitude_violations"
        ]
    )


    print(
        "  Token-limit hit      :",
        generation[
            "hit_token_limit"
        ]
    )


    print(
        "\nAudit:",
        (
            "PASSED"
            if audit[
                "audit_pass"
            ]
            else "FAILED"
        )
    )


# ============================================================
# SAVE JSON
# ============================================================

with open(

    FIRST10_JSON,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        first10_results,

        f,

        indent=2,

        ensure_ascii=False
    )


# ============================================================
# SAVE CSV
# ============================================================

first10_df = pd.DataFrame([

    {

        **{

            key:
                value

            for key, value
            in result.items()

            if key
            != "audit_details"
        },


        "audit_details":
            json.dumps(

                result[
                    "audit_details"
                ],

                ensure_ascii=False
            )
    }

    for result
    in first10_results
])


first10_df.to_csv(

    FIRST10_CSV,

    index=False
)


# ============================================================
# BATCH SUMMARY
# ============================================================

n_generated = len(
    first10_results
)


n_pass = sum(

    bool(
        result[
            "audit_pass"
        ]
    )

    for result
    in first10_results
)


n_fail = (
    n_generated
    -
    n_pass
)


n_token_limit = sum(

    bool(
        result[
            "hit_token_limit"
        ]
    )

    for result
    in first10_results
)


n_top2_fail = sum(

    not bool(
        result[
            "dominant_top2_fidelity_ok"
        ]
    )

    for result
    in first10_results
)


n_top2_sum_fail = sum(

    not bool(
        result[
            "dominant_top2_sum_present"
        ]
    )

    for result
    in first10_results
)


n_interpretation_fail = sum(

    not bool(
        result[
            "dominant_interpretation_present"
        ]
    )

    for result
    in first10_results
)


n_role_overclaim = sum(

    int(
        result[
            "role_overclaim_violation_count"
        ]
        > 0
    )

    for result
    in first10_results
)


n_behavioural = sum(

    int(
        result[
            "behavioural_violation_count"
        ]
        > 0
    )

    for result
    in first10_results
)


n_causal = sum(

    int(
        result[
            "causal_violation_count"
        ]
        > 0
    )

    for result
    in first10_results
)


n_magnitude = sum(

    int(
        result[
            "magnitude_violation_count"
        ]
        > 0
    )

    for result
    in first10_results
)


print(
    "\n" + "=" * 90
)


print(
    "FIRST 10 BATCH SUMMARY"
)


print(
    "=" * 90
)


print(
    "Generated                    :",
    n_generated
)


print(
    "Audit passed                 :",
    n_pass
)


print(
    "Audit failed                 :",
    n_fail
)


print(
    "Top-2 fidelity failures      :",
    n_top2_fail
)


print(
    "Top-2 sum failures           :",
    n_top2_sum_fail
)


print(
    "Interpretation failures      :",
    n_interpretation_fail
)


print(
    "Role-overclaim respondents   :",
    n_role_overclaim
)


print(
    "Behaviour-violation resp.    :",
    n_behavioural
)


print(
    "Causal-violation respondents :",
    n_causal
)


print(
    "Magnitude-violation resp.    :",
    n_magnitude
)


print(
    "Token-limit hits             :",
    n_token_limit
)


# ============================================================
# DOMINANT-ROLE DISTRIBUTION IN PILOT
# ============================================================

if n_generated > 0:

    print(
        "\nPilot dominant-role counts:"
    )


    print(

        first10_df[
            "dominant_role"
        ]

        .value_counts()
    )


# ============================================================
# FINAL PILOT DECISION
# ============================================================

print(
    "\n" + "=" * 90
)


if (
    n_pass == n_generated
    and
    n_token_limit == 0
):

    print(
        "FIRST-10 PILOT STATUS: READY FOR FULL BATCH"
    )


    print(
        "All generated explanations passed the "
        "strict Local Top-2 fidelity audit."
    )


else:

    print(
        "FIRST-10 PILOT STATUS: NOT READY FOR FULL BATCH"
    )


    print(
        "Inspect failed respondents before generating "
        "all 348 explanations."
    )


print(
    "=" * 90
)


print(
    "\nSaved:"
)


print(
    FIRST10_JSON
)


print(
    FIRST10_CSV
)


QWEN3 — FIRST 10 RESPONDENTS
STRICT LOCAL TOP-2 DOMINANT-ROLE AUDIT

[1/10] Respondent 0
Expected dominant role: Economic
Expected Top-2:
  Rank 1 | Average Expendature for transportation | SHAP +0.083161 | |SHAP| 0.083161
  Rank 2 | Monthly Income | SHAP +0.061585 | |SHAP| 0.061585
Expected Top-2 sum: 0.144746

Predicted mode:
Intermediate

Prediction probability:
0.76

Runner-up mode:
Public

Runner-up probability:
0.17

Probability margin:
0.59

Supporting evidence:
- Feature=Average Expendature for transportation | Value=1000 BDT/month | SHAP=+0.083161 | Direction=toward predicted class
- Feature=Monthly Income | Value=0 BDT/month | SHAP=+0.061585 | Direction=toward predicted class
- Feature=Age | Value=22 years | SHAP=+0.043858 | Direction=toward predicted class
- Feature=Effect of Cost on Mode Choice | Value=Yes | SHAP=+0.042241 | Direction=toward predicted class
- Feature=Mode_When_Depressed | Value=Intermediate (Easybike/Rickshaw) | SHAP=+0.038284 | Direction=toward predicted c

## Full 348-respondent generation and audit

This final cell can generate all 348 respondent explanations. Each explanation is generated once with deterministic decoding and audited once. Failed audits are **retained as failures** rather than silently regenerated, preserving the audit as genuine quality-control evidence.

Set `RUN_FULL_BATCH = False` if you only want the respondent-0 / first-10 QC cells.


In [ ]:
# ============================================================
# FULL QWEN BATCH — CHECKPOINTED CHUNK SETUP
# ============================================================
# Run this setup cell once after loading the model/evidence.
# Then run the three chunk cells below separately.
#
# IMPORTANT:
# - Results are saved after EVERY respondent.
# - If Colab/runtime disconnects, rerun the notebook setup/model cells,
#   then rerun the interrupted chunk cell.
# - Already-saved respondent IDs are automatically skipped.
# ============================================================

import os, json, time, gc
import pandas as pd
import torch

FULL_JSON = os.path.join(OUTPUT_DIR, "qwen_all348_explanations_audited.json")
FULL_JSONL = os.path.join(OUTPUT_DIR, "qwen_all348_explanations_audited.jsonl")
FULL_CSV = os.path.join(OUTPUT_DIR, "qwen_all348_explanations_audited.csv")
CHECKPOINT_JSON = os.path.join(OUTPUT_DIR, "qwen_all348_checkpoint.json")
SUMMARY_JSON = os.path.join(OUTPUT_DIR, "qwen_all348_audit_summary.json")


def json_default(obj):
    try:
        import numpy as np
        if isinstance(obj, np.generic):
            return obj.item()
        if isinstance(obj, np.ndarray):
            return obj.tolist()
    except Exception:
        pass
    if isinstance(obj, set):
        return list(obj)
    return str(obj)


def compact_batch_record(item, generation, audit, error=None):
    dom = item.get("dominant_role", {})
    return {
        "respondent_id": item.get("respondent_id"),
        "predicted_mode": item.get("model_output", {}).get("predicted_mode"),
        "prediction_probability": item.get("model_output", {}).get("prediction_probability"),
        "runner_up_mode": item.get("model_output", {}).get("runner_up_mode"),
        "runner_up_probability": item.get("model_output", {}).get("runner_up_probability"),
        "probability_margin": item.get("model_output", {}).get("probability_margin"),
        "dominant_role": dom.get("role"),
        "dominant_role_top2_absolute_shap_sum": dom.get("top2_absolute_shap_sum"),
        "response": generation.get("response") if generation else None,
        "generated_tokens": generation.get("generated_tokens") if generation else None,
        "generation_seconds": generation.get("generation_seconds") if generation else None,
        "tokens_per_second": generation.get("tokens_per_second") if generation else None,
        "hit_token_limit": generation.get("hit_token_limit") if generation else None,
        "audit_pass": audit.get("audit_pass") if audit else False,
        "missing_sections": audit.get("missing_sections") if audit else [],
        "causal_violations": audit.get("causal_violations") if audit else [],
        "role_overclaims": audit.get("role_overclaims") if audit else [],
        "behavioural_violations": audit.get("behavioural_violations") if audit else [],
        "magnitude_violations": audit.get("magnitude_violations") if audit else [],
        "uncertainty_violations": audit.get("uncertainty_violations") if audit else [],
        "error": error,
        "full_audit": audit,
    }


def save_batch(records):
    # Keep deterministic respondent order in all saved outputs.
    records = sorted(records, key=lambda r: int(r["respondent_id"]))

    with open(FULL_JSON, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False, default=json_default)

    with open(FULL_JSONL, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False, default=json_default) + "\n")

    flat = []
    for rec in records:
        flat.append({
            "respondent_id": rec["respondent_id"],
            "predicted_mode": rec["predicted_mode"],
            "prediction_probability": rec["prediction_probability"],
            "runner_up_mode": rec["runner_up_mode"],
            "runner_up_probability": rec["runner_up_probability"],
            "probability_margin": rec["probability_margin"],
            "dominant_role": rec["dominant_role"],
            "dominant_role_top2_absolute_shap_sum": rec["dominant_role_top2_absolute_shap_sum"],
            "response": rec["response"],
            "generated_tokens": rec["generated_tokens"],
            "generation_seconds": rec["generation_seconds"],
            "tokens_per_second": rec["tokens_per_second"],
            "hit_token_limit": rec["hit_token_limit"],
            "audit_pass": rec["audit_pass"],
            "missing_sections": json.dumps(rec["missing_sections"], ensure_ascii=False),
            "causal_violations": json.dumps(rec["causal_violations"], ensure_ascii=False),
            "role_overclaims": json.dumps(rec["role_overclaims"], ensure_ascii=False),
            "behavioural_violations": json.dumps(rec["behavioural_violations"], ensure_ascii=False),
            "magnitude_violations": json.dumps(rec["magnitude_violations"], ensure_ascii=False),
            "uncertainty_violations": json.dumps(rec["uncertainty_violations"], ensure_ascii=False),
            "error": rec["error"],
        })

    pd.DataFrame(flat).to_csv(FULL_CSV, index=False)


def load_saved_records():
    if not os.path.exists(FULL_JSON):
        return []

    try:
        with open(FULL_JSON, "r", encoding="utf-8") as f:
            saved = json.load(f)
        if not isinstance(saved, list):
            raise ValueError("FULL_JSON does not contain a list.")
        return saved
    except Exception as exc:
        raise RuntimeError(
            f"Could not safely recover existing results from {FULL_JSON}: {exc}"
        ) from exc


def write_checkpoint(records, chunk_label, last_position=None):
    checkpoint = {
        "chunk": chunk_label,
        "last_position": last_position,
        "saved_records": len(records),
        "total_respondents": len(evidence_data),
        "saved_respondent_ids": sorted(
            [int(r["respondent_id"]) for r in records]
        ),
        "updated_unix_time": time.time(),
    }
    with open(CHECKPOINT_JSON, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=2, ensure_ascii=False, default=json_default)


def run_qwen_chunk(start_idx, end_idx, chunk_label):
    """Run one zero-based slice [start_idx:end_idx], resuming from Drive."""

    end_idx = min(end_idx, len(evidence_data))
    records = load_saved_records()

    # De-duplicate any legacy/recovered records by respondent_id.
    by_id = {str(r["respondent_id"]): r for r in records}
    records = list(by_id.values())
    completed_ids = set(by_id.keys())

    print("=" * 90)
    print(chunk_label)
    print(f"Target positions : {start_idx + 1}–{end_idx}")
    print(f"Already saved    : {len(records)}")
    print("=" * 90)

    chunk_start = time.time()

    for pos in range(start_idx, end_idx):
        item = evidence_data[pos]
        rid = str(item["respondent_id"])

        if rid in completed_ids:
            print(f"[SKIP {pos+1:03d}/{len(evidence_data)}] respondent={rid} already saved")
            continue

        generation = None
        audit = None
        error = None

        try:
            generation = generate_qwen_explanation(item)
            audit = audit_qwen_explanation(item, generation)
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"

        rec = compact_batch_record(item, generation, audit, error)
        records.append(rec)
        completed_ids.add(rid)

        status = "PASS" if rec["audit_pass"] else "FAIL"
        print(
            f"[{pos+1:03d}/{len(evidence_data)}] "
            f"respondent={rid} audit={status}"
        )

        # Critical safety feature: persist after EVERY respondent.
        save_batch(records)
        write_checkpoint(records, chunk_label, last_position=pos + 1)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elapsed = time.time() - chunk_start
    records = load_saved_records()

    print("\n" + "=" * 90)
    print(f"{chunk_label} COMPLETE")
    print(f"Saved records : {len(records)}/{len(evidence_data)}")
    print(f"Chunk seconds : {elapsed:.1f}")
    print("=" * 90)

    return records


def write_final_summary():
    records = load_saved_records()
    n_pass = sum(bool(r.get("audit_pass")) for r in records)
    n_fail = len(records) - n_pass
    n_error = sum(r.get("error") is not None for r in records)
    n_token_limit = sum(
        bool(r.get("hit_token_limit"))
        for r in records
        if r.get("hit_token_limit") is not None
    )

    summary = {
        "generated": len(records),
        "expected": len(evidence_data),
        "complete": len(records) == len(evidence_data),
        "audit_passed": n_pass,
        "audit_failed": n_fail,
        "generation_errors": n_error,
        "token_limit_hits": n_token_limit,
        "audit_pass_rate": n_pass / len(records) if records else None,
        "full_json": FULL_JSON,
        "full_jsonl": FULL_JSONL,
        "full_csv": FULL_CSV,
        "checkpoint_json": CHECKPOINT_JSON,
    }

    with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False, default=json_default)

    print("\nFINAL BATCH SUMMARY")
    print(json.dumps(summary, indent=2, ensure_ascii=False, default=json_default))
    return summary


print("Checkpointed chunk runner ready.")
print("Run Chunk 1, Chunk 2, and Chunk 3 in separate cells below.")


Checkpointed chunk runner ready.
Run Chunk 1, Chunk 2, and Chunk 3 in separate cells below.


In [ ]:
# ============================================================
# QWEN CHUNK 1 — RESPONDENTS / POSITIONS 1 TO 120
# ============================================================
chunk1_records = run_qwen_chunk(
    start_idx=0,
    end_idx=120,
    chunk_label="QWEN CHUNK 1 — 1 TO 120",
)


QWEN CHUNK 1 — 1 TO 120
Target positions : 1–120
Already saved    : 0
[001/348] respondent=0 audit=PASS
[002/348] respondent=1 audit=PASS
[003/348] respondent=2 audit=PASS
[004/348] respondent=3 audit=PASS
[005/348] respondent=4 audit=PASS
[006/348] respondent=5 audit=FAIL
[007/348] respondent=6 audit=PASS
[008/348] respondent=7 audit=PASS
[009/348] respondent=8 audit=PASS
[010/348] respondent=9 audit=PASS
[011/348] respondent=10 audit=PASS
[012/348] respondent=11 audit=PASS
[013/348] respondent=12 audit=PASS
[014/348] respondent=13 audit=PASS
[015/348] respondent=14 audit=PASS
[016/348] respondent=15 audit=PASS
[017/348] respondent=16 audit=PASS
[018/348] respondent=17 audit=PASS
[019/348] respondent=18 audit=PASS
[020/348] respondent=19 audit=PASS
[021/348] respondent=20 audit=PASS
[022/348] respondent=21 audit=FAIL
[023/348] respondent=22 audit=PASS
[024/348] respondent=23 audit=PASS
[025/348] respondent=24 audit=PASS
[026/348] respondent=25 audit=PASS
[027/348] respondent=26 audit=

In [ ]:
# ============================================================
# QWEN CHUNK 2 — RESPONDENTS / POSITIONS 121 TO 240
# ============================================================
chunk2_records = run_qwen_chunk(
    start_idx=120,
    end_idx=240,
    chunk_label="QWEN CHUNK 2 — 121 TO 240",
)


QWEN CHUNK 2 — 121 TO 240
Target positions : 121–240
Already saved    : 120
[121/348] respondent=120 audit=PASS
[122/348] respondent=121 audit=PASS
[123/348] respondent=122 audit=PASS
[124/348] respondent=123 audit=PASS
[125/348] respondent=124 audit=PASS
[126/348] respondent=125 audit=PASS
[127/348] respondent=126 audit=PASS
[128/348] respondent=127 audit=PASS
[129/348] respondent=128 audit=PASS
[130/348] respondent=129 audit=PASS
[131/348] respondent=130 audit=PASS
[132/348] respondent=131 audit=PASS
[133/348] respondent=132 audit=PASS
[134/348] respondent=133 audit=PASS
[135/348] respondent=134 audit=PASS
[136/348] respondent=135 audit=FAIL
[137/348] respondent=136 audit=PASS
[138/348] respondent=137 audit=PASS
[139/348] respondent=138 audit=PASS
[140/348] respondent=139 audit=PASS
[141/348] respondent=140 audit=PASS
[142/348] respondent=141 audit=PASS
[143/348] respondent=142 audit=FAIL
[144/348] respondent=143 audit=PASS
[145/348] respondent=144 audit=PASS
[146/348] respondent=145

In [ ]:
# ============================================================
# QWEN CHUNK 3 — RESPONDENTS / POSITIONS 241 TO REMAINING
# ============================================================
chunk3_records = run_qwen_chunk(
    start_idx=240,
    end_idx=len(evidence_data),
    chunk_label=f"QWEN CHUNK 3 — 241 TO {len(evidence_data)}",
)

# Write/update the final audit summary after the last chunk.
final_summary = write_final_summary()


## Saved LLM outputs — checkpointed 3-chunk workflow

The full Qwen generation is now split into three independent execution cells:

- Chunk 1: positions **1–120**
- Chunk 2: positions **121–240**
- Chunk 3: positions **241–remaining**

Results are written to Google Drive **after every respondent**. If the runtime disconnects, rerun the setup/model cells and then rerun the interrupted chunk cell. Existing respondent IDs in `qwen_all348_explanations_audited.json` are automatically skipped.

Outputs remain under:

`/content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/Qwen3 Agentic AI Explanations/`
